## Table Of Contents

5. [Data Cleaning](#5-data-cleaning)
    - [Traffic Stop](#51-traffic-stop)
        - [date_of_stop](#511-date_of_stop)
        - [time_of_stop](#512-time_of_stop)
        - [agency](#513-agency)
        - [sub_agency](#514-sub_agency)
        - [location](#515-location)
        - [latitude & longitude](#516-latitude--longitude)
        - [geolocation](#517-geolocation)
        - [traffic_stop table creation](#518-traffic_stop-table-creation)
    - [Incident_safety](#52-incident_safety)
        - [Standardize categorical columns](#521-standardize-categorical-columns)
    - [Driver_Vehicle](#53-driver_vehicle)
        - [race](#531-race)
        - [gender](#532-gender)
        - [driver statr](#533-driver_state)
        - [dl_state](#534-dl_state)
        - [vehicle_type](#535-vehicle_type)
        - [year](#536-year)
        - [make](#537-make)
        - [model](#538-model)
        - [color](#538-model)
        - [driver_city](#5310-driver_city)
        - [driver_vehicle_df table creation](#5311-driver_vehicle_df-table-creation)
    - [Violation_Charge](#54-violation_charge)
        - [description](#541-description)
        - [violation_type](#542-violation_type)
        - [charge](#543-charge)
        - [article](#544-article)
        - [search_arrest_reason](#545-search_arrest_reason)
        - [violation_charge_df Table creation](#546-violation_charge_df-table-creation)
    - [Search_Enforcement](#55-search_enforcement)
        - [search_conducted](#551-search_conducted)
        - [search_disposition](#552-search_disposition)
        - [search_outcome](#553-search_outcome)
        - [search_reason](#554-search_reason)
        - [search_reason_for_stop](#555-search_reason_for_stop)
        - [search_type](#556-search_type)
        - [arrest_type](#557-arrest_type)
        - [search_enforcement_df Table Creation](#558-search_enforcement_df-table-creation)



# 5. Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sqlalchemy import create_engine
import re
import plotly.express as px


In [2]:
# Make plots look nicer and all plots consistently
sns.set_theme(style="whitegrid")
#Show all columns in output
pd.set_option('display.max_columns',None)

### Load Data

In [3]:
traffic_df=pd.read_csv("../data/traffic_v1_df.csv")

## 5.1 Traffic Stop

#### 5.1.1 date_of_stop

In [4]:
traffic_df['date_of_stop'].nunique()

4821

In [5]:
traffic_df['date_of_stop'].sample(20, random_state=52)

618644             06/28/2017
677838    2017-08-07 00:00:00
708973    2012-06-07 00:00:00
212582             07/25/2015
212839    2014-02-04 00:00:00
283633    2012-01-10 00:00:00
772296             03/15/2018
235074             04/17/2013
488786    2015-07-02 00:00:00
766733             08/20/2014
675642             10/19/2015
714521             11/19/2015
734064             10/22/2022
386489             05/15/2019
726786             03/31/2016
767267             10/17/2018
316607             10/23/2012
655845             12/20/2014
520372             07/24/2015
388668    2017-08-07 00:00:00
Name: date_of_stop, dtype: object

In [6]:
# First, clean strings
traffic_df['date_of_stop_clean'] = traffic_df['date_of_stop'].astype(str).str.strip()


In [7]:
# First pass: infer format automatically
traffic_df['date_of_stop_clean'] = pd.to_datetime(
    traffic_df['date_of_stop_clean'],
    errors='coerce',
    infer_datetime_format=True
)
traffic_df['date_of_stop_clean']

C:\Users\svind\AppData\Local\Temp\ipykernel_22340\2744832214.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  traffic_df['date_of_stop_clean'] = pd.to_datetime(


0         2023-01-05
1                NaT
2                NaT
3                NaT
4                NaT
             ...    
1048570          NaT
1048571          NaT
1048572   2012-08-06
1048573   2016-02-04
1048574          NaT
Name: date_of_stop_clean, Length: 1048575, dtype: datetime64[ns]

In [8]:
# Second pass: fix MM/DD/YYYY that failed in first pass
mask = traffic_df['date_of_stop_clean'].isna()
traffic_df.loc[mask, 'date_of_stop_clean'] = pd.to_datetime(
    traffic_df.loc[mask, 'date_of_stop'].astype(str).str.strip(),
    format='%m/%d/%Y',
    errors='coerce'
)


In [9]:
traffic_df['date_of_stop_clean'].dt.normalize()
traffic_df['date_of_stop_clean']

0         2023-01-05
1         2023-08-31
2         2023-08-31
3         2023-08-31
4         2023-08-31
             ...    
1048570   2015-03-31
1048571   2023-01-13
1048572   2012-08-06
1048573   2016-02-04
1048574   2014-01-15
Name: date_of_stop_clean, Length: 1048575, dtype: datetime64[ns]

In [10]:
traffic_df['date_of_stop_clean'].isna().sum()

np.int64(0)

In [11]:
today = pd.Timestamp.today()
future_dates = traffic_df[traffic_df['date_of_stop_clean'] > today]

print("Future dates:\n", future_dates[['date_of_stop', 'date_of_stop_clean']].head(10))
print("Total future dates:", len(future_dates))

Future dates:
 Empty DataFrame
Columns: [date_of_stop, date_of_stop_clean]
Index: []
Total future dates: 0


In [12]:
invalid_years = traffic_df[
    (traffic_df['date_of_stop_clean'].dt.year < 1900) |
    (traffic_df['date_of_stop_clean'].dt.year > today.year)
]

print("Invalid year dates:\n", invalid_years[['date_of_stop', 'date_of_stop_clean']].head(10))
print("Total invalid year rows:", len(invalid_years))

Invalid year dates:
 Empty DataFrame
Columns: [date_of_stop, date_of_stop_clean]
Index: []
Total invalid year rows: 0


In [13]:
traffic_df['date_of_stop'] = traffic_df['date_of_stop_clean']


In [14]:
traffic_df.drop(columns=['date_of_stop_clean'], inplace=True)

#### 5.1.2 time_of_stop

In [15]:
traffic_df['time_of_stop'].sample(20, random_state=30)

817193     03:32:00
438441     21:53:00
370515     22:41:00
840250     14:25:00
250809     22:43:00
875526     18:01:00
190066     00:20:00
365413     22:28:00
272841     20:07:00
288768     10:45:00
63305      06:53:00
278872     22:32:00
655558     23:47:00
1015853    09:04:00
173947     01:10:00
1017141    08:00:00
548219     09:54:00
1042338    12:34:00
558572     13:58:00
658534     23:58:00
Name: time_of_stop, dtype: object

In [16]:
# Replace dots with colons
traffic_df['time_of_stop_clean'] = traffic_df['time_of_stop'].str.replace('.', ':')

In [17]:
# Convert to datetime.time
traffic_df['time_of_stop_clean'] = pd.to_datetime(
    traffic_df['time_of_stop_clean'], format='%H:%M:%S', errors='coerce'
).dt.time

In [18]:
# Check for any parsing issues
traffic_df['time_of_stop_clean'].isna().sum()

np.int64(0)

In [19]:
traffic_df['time_of_stop'] = traffic_df['time_of_stop_clean']


In [20]:
traffic_df.drop(columns=['time_of_stop_clean'], inplace=True)

In [21]:
# Combine cleaned date and time into a full timestamp
traffic_df['stop_timestamp'] = pd.to_datetime(
    traffic_df['date_of_stop'].astype(str) + ' ' + 
    traffic_df['time_of_stop'].astype(str),
    errors='coerce'
)

In [22]:
traffic_df['stop_timestamp'].sample(20,random_state=42)

781974    2017-03-23 08:21:00
937737    2014-09-04 23:08:00
907828    2017-05-31 12:32:00
784628    2019-04-01 08:43:00
662460    2014-10-01 00:28:00
280139    2015-04-04 03:41:00
355572    2015-02-10 23:21:00
749979    2013-05-21 10:17:00
374753    2017-07-30 20:43:00
17327     2023-05-31 07:07:00
658472    2016-06-09 03:09:00
700377    2012-05-07 08:37:00
880383    2018-08-19 03:22:00
677181    2018-07-15 23:13:00
18941     2023-12-20 06:30:00
735693    2016-10-07 00:41:00
1034300   2023-04-26 07:17:00
389058    2022-03-19 02:02:00
473202    2015-04-30 04:28:00
677891    2018-12-28 23:52:00
Name: stop_timestamp, dtype: datetime64[ns]

#### 5.1.3 agency

In [23]:
# Strip whitespace and convert to uppercase
traffic_df['agency_clean'] = traffic_df['agency'].astype(str).str.strip().str.upper()

In [24]:
print(traffic_df['agency_clean'].value_counts())

agency_clean
MCP    1048575
Name: count, dtype: int64


In [25]:
traffic_df['agency']=traffic_df['agency_clean']

#### 5.1.4 sub_agency

In [26]:
# Strip whitespace
traffic_df['sub_agency_clean'] = traffic_df['sub_agency'].astype(str).str.strip()

# Standardize capitalization
traffic_df['sub_agency_clean'] = traffic_df['sub_agency_clean'].str.title()


In [27]:

# Optional: normalize spacing/punctuation
traffic_df['sub_agency_clean'] = traffic_df['sub_agency_clean'].str.replace(r'\s+', ' ', regex=True)
traffic_df['sub_agency_clean'] = traffic_df['sub_agency_clean'].str.replace(r'\s?,\s?', ', ', regex=True)



In [28]:
print(traffic_df['sub_agency_clean'].nunique())


9


In [29]:
print(traffic_df['sub_agency_clean'].sample(20, random_state=42))

781974                             1St District, Rockville
937737                             1St District, Rockville
907828                            5Th District, Germantown
784628     6Th District, Gaithersburg / Montgomery Village
662460                              2Nd District, Bethesda
280139     6Th District, Gaithersburg / Montgomery Village
355572                              2Nd District, Bethesda
749979                              2Nd District, Bethesda
374753                         3Rd District, Silver Spring
17327                  Headquarters And Special Operations
658472     6Th District, Gaithersburg / Montgomery Village
700377                         3Rd District, Silver Spring
880383                 Headquarters And Special Operations
677181                               4Th District, Wheaton
18941                  Headquarters And Special Operations
735693                             1St District, Rockville
1034300                Headquarters And Special Operatio

In [30]:
# Convert the ordinal suffixes in district names to lowercase
traffic_df['sub_agency_clean'] = traffic_df['sub_agency_clean'].str.replace(
    r'(\d+)(St|Nd|Rd|Th)\b', lambda m: m.group(1) + m.group(2).lower(), regex=True
)

# Check the result
print(traffic_df['sub_agency_clean'].sample(20, random_state=52))

618644                        3rd District, Silver Spring
677838                              4th District, Wheaton
708973                        3rd District, Silver Spring
212582                              4th District, Wheaton
212839                              4th District, Wheaton
283633                             2nd District, Bethesda
772296    6th District, Gaithersburg / Montgomery Village
235074                              4th District, Wheaton
488786                             2nd District, Bethesda
766733                              4th District, Wheaton
675642                        3rd District, Silver Spring
714521                Headquarters And Special Operations
734064                           5th District, Germantown
386489                             2nd District, Bethesda
726786                           5th District, Germantown
767267                        3rd District, Silver Spring
316607                              4th District, Wheaton
655845        

In [31]:
traffic_df['sub_agency_clean'].value_counts()

sub_agency_clean
4th District, Wheaton                              223650
3rd District, Silver Spring                        192627
2nd District, Bethesda                             166680
5th District, Germantown                           140466
6th District, Gaithersburg / Montgomery Village    116567
1st District, Rockville                            116083
Headquarters And Special Operations                 92500
W15                                                     1
S15                                                     1
Name: count, dtype: int64

In [32]:
# Map W15 and S15 to Headquarters
traffic_df['sub_agency_clean'] = traffic_df['sub_agency_clean'].replace({'W15': 'Headquarters And Special Operations',
                                                                       'S15': 'Headquarters And Special Operations'})

In [33]:
traffic_df['sub_agency_clean'].value_counts()

sub_agency_clean
4th District, Wheaton                              223650
3rd District, Silver Spring                        192627
2nd District, Bethesda                             166680
5th District, Germantown                           140466
6th District, Gaithersburg / Montgomery Village    116567
1st District, Rockville                            116083
Headquarters And Special Operations                 92502
Name: count, dtype: int64

In [34]:
# Optional: extract district number using regex
traffic_df['district_number'] = traffic_df['sub_agency_clean'].str.extract(r'(\d+)(?:[st|nd|rd|th] District)?')[0]

In [35]:
# Replace original sub_agency with cleaned version
traffic_df['sub_agency'] = traffic_df['sub_agency_clean']

# Drop the now redundant columns
traffic_df = traffic_df.drop(columns=['sub_agency_clean', 'agency_clean'])

In [36]:
traffic_df['district_number'].value_counts()

district_number
4    223650
3    192627
2    166680
5    140466
6    116567
1    116083
Name: count, dtype: int64

#### 5.1.5 location

In [37]:
"""
Montgomery County Traffic Location Cleaning Pipeline
=====================================================
Cleans and standardizes the 'Location' column into:
  - loc       : normalized location string
  - road_1    : first road name
  - road_2    : second road name (intersection partner)
  - direction : NB / SB / EB / WB
  - exit_ref  : EXT/EXIT reference if present
  - loc_type  : intersection / address / single_road
  - is_landmark : True if road_2 looks like a landmark
  - needs_review: True if row needs manual review
"""

"\nMontgomery County Traffic Location Cleaning Pipeline\n=====================================================\nCleans and standardizes the 'Location' column into:\n  - loc       : normalized location string\n  - road_1    : first road name\n  - road_2    : second road name (intersection partner)\n  - direction : NB / SB / EB / WB\n  - exit_ref  : EXT/EXIT reference if present\n  - loc_type  : intersection / address / single_road\n  - is_landmark : True if road_2 looks like a landmark\n  - needs_review: True if row needs manual review\n"

In [38]:

# ══════════════════════════════════════════════════════════════════════
# CONSTANTS — defined once, reused across all steps
# ══════════════════════════════════════════════════════════════════════

# road type pattern — no capture group, used for detection
ROAD_TYPE_PAT = (
    r'\b(?:BLVD|AVE|AV|ST|RD|DR|LN|PKWY|HWY|WAY|PL|CT|TER|CIR'
    r'|ROAD|STREET|AVENUE|DRIVE|LANE|PLACE|COURT|TERRACE|PARKWAY'
    r'|PIKE|FWY|FREEWAY|EXPY|EXPRESSWAY|TURNPIKE|TPKE)\b'
)

# road type pattern — with capture group, used for substitution
ROAD_TYPE_CAP = (
    r'(BLVD|AVE|AV|ST|RD|DR|LN|PKWY|HWY|WAY|PL|CT|TER|CIR'
    r'|ROAD|STREET|AVENUE|DRIVE|LANE|PLACE|COURT|TERRACE|PARKWAY'
    r'|PIKE|FWY|FREEWAY|EXPY|EXPRESSWAY|TURNPIKE|TPKE)'
)

# route pattern — e.g. I-270, MD-355, US-29
ROUTE_PAT = r'^(?:I|RT|US|MD|SR|HWY|RTE)[-\s]?\d+[A-Z]?$'

# direction abbreviations
DIR = r'\b(NB|SB|EB|WB)\b'

# road type normalization map — full words to abbreviations
ROAD_TYPE_NORM = [
    (r'\bDRIVE\b',     'DR'),   (r'\bAVENUE\b',    'AVE'),
    (r'\bROAD\b',      'RD'),   (r'\bSTREET\b',     'ST'),
    (r'\bLANE\b',      'LN'),   (r'\bCOURT\b',      'CT'),
    (r'\bPLACE\b',     'PL'),   (r'\bTERRACE\b',    'TER'),
    (r'\bPARKWAY\b',  'PKWY'),  (r'\bHIGHWAY\b',   'HWY'),
    (r'\bBOULEVARD\b','BLVD'),  (r'\bCIRCLE\b',    'CIR'),
    (r'\bFREEWAY\b',  'FWY'),   (r'\bTURNPIKE\b', 'TPKE'),
    (r'\bEXPRESSWAY\b','EXPY'),
]

# cardinal word normalization map
CARDINAL_NORM = [
    (r'\bEAST\b',  'E.'), (r'\bWEST\b',  'W.'),
    (r'\bNORTH\b', 'N.'), (r'\bSOUTH\b', 'S.'),
]

# direction word normalization map
DIRECTION_MAP = [
    (r'\bWEST\s*BOUND\b',  'WB'), (r'\bEAST\s*BOUND\b',  'EB'),
    (r'\bNORTH\s*BOUND\b', 'NB'), (r'\bSOUTH\s*BOUND\b', 'SB'),
    (r'\bW/B\b', 'WB'), (r'\bE/B\b', 'EB'),
    (r'\bN/B\b', 'NB'), (r'\bS/B\b', 'SB'),
]

In [39]:

# ══════════════════════════════════════════════════════════════════════
# STEP 1 — uppercase + collapse whitespace
# ══════════════════════════════════════════════════════════════════════
traffic_df['loc'] = (
    traffic_df['location']
    .str.upper()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)


In [40]:

# ══════════════════════════════════════════════════════════════════════
# STEP 2 — normalize direction words + extract to 'direction' column
# ══════════════════════════════════════════════════════════════════════
traffic_df['direction'] = traffic_df['loc'].str.extract(DIR, expand=False)

traffic_df['loc'] = traffic_df['loc'].str.replace(
    r'\b(WEST\s*BOUND|EAST\s*BOUND|NORTH\s*BOUND|SOUTH\s*BOUND|W/B|E/B|N/B|S/B)\b',
    lambda m: {
        'W/B': 'WB', 'E/B': 'EB', 'N/B': 'NB', 'S/B': 'SB'
    }.get(m.group(0), m.group(0)[0] + 'B'),
    regex=True
)


In [41]:

# ══════════════════════════════════════════════════════════════════════
# STEP 2c — extract EXIT / EXT / OFFRAMP refs before split
# ══════════════════════════════════════════════════════════════════════
traffic_df['exit_ref'] = traffic_df['loc'].str.extract(
    r'((?:EXIT|EXT|OFFRAMP|OFF\s*RAMP)\s*\d*)', expand=False
)
traffic_df['loc'] = traffic_df['loc'].str.replace(
    r'\s*/\s*(?:EXIT|EXT|OFFRAMP|OFF\s*RAMP)\s*\d*', '', regex=True
).str.strip()


In [42]:


# ══════════════════════════════════════════════════════════════════════
# STEP 2d — encode L/N lane designator (e.g. RT 270 L/N 2)
# ══════════════════════════════════════════════════════════════════════
traffic_df['loc'] = traffic_df['loc'].str.replace(
    r'\bL/N\s*(\d+)?\b',
    lambda m: 'LNLANE' + (m.group(1) or ''),
    regex=True
)


In [43]:


# ══════════════════════════════════════════════════════════════════════
# STEP 2e — encode E W HWY (East-West Highway)
# ══════════════════════════════════════════════════════════════════════
traffic_df['loc'] = traffic_df['loc'].str.replace(
    r'\bE\.?\s*W\.?\s*HWY\b', 'EWHWY_ENCODED', regex=True
)


In [44]:

# ══════════════════════════════════════════════════════════════════════
# STEP 3 — normalize all delimiters to "/"
# ══════════════════════════════════════════════════════════════════════
DELIM_NORM = r'\s*(?:\bAT\b|\bAND\b|\bON\b|\bX\b|[@&]|(?<=BLK) OF)\s*'
traffic_df['loc'] = traffic_df['loc'].str.replace(DELIM_NORM, '/', regex=True)


In [45]:


# ══════════════════════════════════════════════════════════════════════
# STEP 3b — strip DIR tags now that "/" boundaries are clear
# ══════════════════════════════════════════════════════════════════════
traffic_df['loc'] = (
    traffic_df['loc']
    .str.replace(r'^' + DIR + r'\s*',         '', regex=True)  # leading
    .str.replace(r'\s*' + DIR + r'\s*(?=/)',   '', regex=True)  # before /
    .str.replace(r'(?<=/)\s*' + DIR + r'\s*',  '', regex=True)  # after /
    .str.replace(r'\s*' + DIR + r'\s*$',       '', regex=True)  # trailing
    .str.replace(r'\s*,\s*' + DIR,             '', regex=True)  # after comma
    .str.replace(r'^[,.\s]+|[,.\s]+$',         '', regex=True)  # punctuation
    .str.strip()
)

# fix stray BWB prefix
traffic_df['loc'] = traffic_df['loc'].str.replace(r'\bBWB\b\s*', '', regex=True).str.strip()


In [46]:


# ══════════════════════════════════════════════════════════════════════
# STEP 4 — split on "/"
# ══════════════════════════════════════════════════════════════════════
traffic_df[['road_1', 'road_2']] = (
    traffic_df['loc']
    .str.split(r'\s*/\s*', n=1, expand=True)
    .fillna('')
    .apply(lambda col: col.str.strip())
)


In [47]:


# ══════════════════════════════════════════════════════════════════════
# STEP 4b — restore encoded values + strip city/state from road_1/road_2
# ══════════════════════════════════════════════════════════════════════

# restore L/N lane designator
for col in ['loc', 'road_1', 'road_2']:
    traffic_df[col] = traffic_df[col].str.replace(
        r'LNLANE(\d*)',
        lambda m: 'L/N ' + m.group(1) if m.group(1) else 'L/N',
        regex=True
    )

# restore E-W HWY
for col in ['loc', 'road_1', 'road_2']:
    traffic_df[col] = traffic_df[col].str.replace('EWHWY_ENCODED', 'E-W HWY', regex=False)

# strip city/state/zip suffix — safe to use \d{0,5} after split
for col in ['road_1', 'road_2']:
    traffic_df[col] = traffic_df[col].str.replace(
        r',\s*(?:[A-Z\s]+,\s*)?(?:MD|VA|DC|WV)\b\s*\d{0,5}\s*$',
        '', regex=True
    ).str.strip()


In [48]:


# ══════════════════════════════════════════════════════════════════════
# STEP 5 — normalize + clean road names
# ══════════════════════════════════════════════════════════════════════

# 5a: fix merged direction+road e.g. "EBGUDE DR" → "GUDE DR"
for col in ['road_1', 'road_2']:
    traffic_df[col] = traffic_df[col].str.replace(
        r'^(EB|WB|NB|SB)(?=[A-Z])', '', regex=True
    )

# 5b: normalize road type full words → abbreviations
for pat, repl in ROAD_TYPE_NORM:
    for col in ['road_1', 'road_2']:
        traffic_df[col] = traffic_df[col].str.replace(pat, repl, regex=True)

# 5c: normalize cardinal words EAST/WEST → E./W.
for pat, repl in CARDINAL_NORM:
    for col in ['road_1', 'road_2']:
        traffic_df[col] = traffic_df[col].str.replace(pat, repl, regex=True)

# 5d: strip periods from road type abbreviations + clean punctuation
for col in ['road_1', 'road_2']:
    traffic_df[col] = (
       traffic_df[col]
        .str.replace(ROAD_TYPE_CAP + r'\.', r'\1',      regex=True)  # BLVD. → BLVD
        .str.replace(r'\b(RT|RTE)\.', r'\1',             regex=True)  # RT.   → RT
        .str.replace(r'\bBOUND\s+[NSEW]\b', '',          regex=True)  # BOUND E artifact
        .str.replace(r'^\b[NSEW]\.\s+[NSEW]\b$', '',     regex=True)  # E. W  artifact
        .str.replace(r'[.,\s]+$|^[.,\s]+', '',           regex=True)  # punctuation
        .str.strip()
    )


In [49]:

# ══════════════════════════════════════════════════════════════════════
# STEP 6 — standardize highway route formats
# ══════════════════════════════════════════════════════════════════════
def standardize_route(text):
    if not isinstance(text, str) or text == '':
        return text
    # I-270, I-270 N
    text = re.sub(
        r'\bI[-\s]?(\d+[A-Z]?)(\s+[NSEW])?\b',
        lambda m: f'I-{m.group(1)}{m.group(2) or ""}',
        text
    )
    # US-29, US-29 N
    text = re.sub(
        r'\bUS[-\s]?(\d+[A-Z]?)(\s+[NSEW])?\b',
        lambda m: f'US-{m.group(1)}{m.group(2) or ""}',
        text
    )
    # MD-355, MD-355 N (RT/RTE/MD/SR all → MD)
    text = re.sub(
        r'\b(?:RT|RTE|MD|SR|HWY)[-\s]?(\d+[A-Z]?)(\s+[NSEW])?\b',
        lambda m: f'MD-{m.group(1)}{m.group(2) or ""}',
        text
    )
    # bare 2-3 digit number → MD-xxx
    text = re.sub(
        r'^\s*(\d{2,3}[A-Z]?)(\s+[NSEW])?\s*$',
        lambda m: f'MD-{m.group(1)}{m.group(2) or ""}',
        text
    )
    return text

for col in ['road_1', 'road_2']:
    traffic_df[col] = traffic_df[col].apply(standardize_route)



In [50]:

# ══════════════════════════════════════════════════════════════════════
# STEP 7 — classify loc_type
# ══════════════════════════════════════════════════════════════════════
traffic_df['road_2'] = traffic_df['road_2'].fillna('').str.strip()

no_road2  = traffic_df['road_2'].eq('')
has_road2 = traffic_df['road_2'].ne('')

is_address = (
    no_road2
    & traffic_df['road_1'].str.match(r'^\d+', na=False)
    & (
        traffic_df['road_1'].str.contains(r'\bBLOCK\b|\bBLK\b', regex=True, na=False)
        | traffic_df['road_1'].str.contains(ROAD_TYPE_PAT, regex=True, na=False)
        | (traffic_df['road_1'].str.split().str.len() >= 4)
    )
)

is_landmark = (
    has_road2
    & ~traffic_df['road_2'].str.contains(ROAD_TYPE_PAT, regex=True, na=False)
    & (traffic_df['road_2'].str.split().str.len() == 1)
)

# ORDER MATTERS — np.select picks first True condition
conditions = [has_road2, is_address, no_road2]
choices    = ['intersection', 'address', 'single_road']
traffic_df['loc_type']    = np.select(conditions, choices, default='unknown')
traffic_df['is_landmark'] = is_landmark



In [51]:

# ══════════════════════════════════════════════════════════════════════
# STEP 8 — needs_review flag
# ══════════════════════════════════════════════════════════════════════
is_valid_route = (
    traffic_df['road_1'].str.match(ROUTE_PAT, na=False)
    | traffic_df['road_2'].str.match(ROUTE_PAT, na=False)
)

traffic_df['needs_review'] = (
    # bare number, not a valid route
    (traffic_df['road_1'].str.match(r'^\d+$', na=False) & ~is_valid_route)
    # very short single word, not a valid route
    | ((traffic_df['road_1'].str.split().str.len() == 1) & (traffic_df['road_1'].str.len() <= 4) & ~is_valid_route)
    # malformed: word then number e.g. "SHADYGROVE 355"
    | (traffic_df['road_1'].str.match(r'^[A-Z].*\s\d+$', na=False) & ~traffic_df['road_1'].str.match(ROUTE_PAT, na=False))
    # no road type and not a route — incomplete road name
    | (
        traffic_df['road_1'].ne('')
        & ~traffic_df['road_1'].str.contains(ROAD_TYPE_PAT, regex=True, na=False)
        & ~traffic_df['road_1'].str.match(ROUTE_PAT, na=False)
        & ~is_valid_route
    )
    # landmark in road_2
    | is_landmark
)


In [52]:
traffic_df['location_clean']=traffic_df['loc']

traffic_df.drop(columns='loc',inplace=True)

In [53]:


# ══════════════════════════════════════════════════════════════════════
# FINAL OUTPUT COLUMNS
# ══════════════════════════════════════════════════════════════════════
print(traffic_df[[
    'location', 'location_clean', 'road_1', 'road_2',
    'direction', 'exit_ref', 'loc_type',
    'is_landmark', 'needs_review'
]].sample(10,random_state=30))

print("\n=== loc_type distribution ===")
print(traffic_df['loc_type'].value_counts())

print("\n=== needs_review count ===")
print(traffic_df['needs_review'].sum(), 'rows flagged for review')

print("\n=== is_landmark count ===")
print(traffic_df['is_landmark'].sum(), 'rows flagged as landmark')

                                    location  \
817193              IL 495 @ COLESVILLE ROAD   
438441  LOST KNIFE RD/MONTGOMERY VILLAGE AVE   
370515   OLD GEORGETOWN RD @ HUNTINGTON PKWY   
840250        VEIRS MILL RD @ ROCKVILLE PIKE   
250809                    11499 STEWART LANE   
875526                      187/WOODMONT AVE   
190066   CALVERTON BOULEVARD AT GALWAY DRIVE   
365413      PRINCE PHILLIP DR @ SPARTAN ROAD   
272841        PLUM ORCHARD AT CHERRY HILL RD   
288768        CLARA BARTON PARKWAY AT LOCK 6   

                              location_clean             road_1  \
817193                IL 495/COLESVILLE ROAD             IL 495   
438441  LOST KNIFE RD/MONTGOMERY VILLAGE AVE      LOST KNIFE RD   
370515     OLD GEORGETOWN RD/HUNTINGTON PKWY  OLD GEORGETOWN RD   
840250          VEIRS MILL RD/ROCKVILLE PIKE      VEIRS MILL RD   
250809                    11499 STEWART LANE   11499 STEWART LN   
875526                      187/WOODMONT AVE             MD-187   
19

In [54]:
traffic_df[traffic_df['location_clean'].isnull()]

,seq_id,date_of_stop,time_of_stop,agency,sub_agency,description,location,latitude,longitude,accident,belts,personal_injury,property_damage,fatal,commercial_license,hazmat,commercial_vehicle,alcohol,work_zone,search_conducted,search_disposition,search_outcome,search_reason,search_reason_for_stop,search_type,search_arrest_reason,state,vehicle_type,year,make,model,color,violation_type,charge,article,contributed_to_accident,race,gender,driver_city,driver_state,dl_state,arrest_type,geolocation,stop_timestamp,district_number,direction,exit_ref,road_1,road_2,loc_type,is_landmark,needs_review,location_clean
614118,98a34f6c-ad4e-4cdd-bf09-bcc1ad19cf73,2013-08-30,23:39:00,MCP,Headquarters And Special Operations,DRIVING VEHICLE ON HIGHWAY WITH SUSPENDED REGI...,NaN,39.006675,-77.07532,No,No,No,No,No,No,No,No,No,No,No,NaN,Citation,NaN,61*,NaN,NaN,MD,02 - Automobile,1997.0,NISSAN,PATHFINDER,RED,Citation,13-401(h),Transportation Article,False,WHITE,M,BETHESDA,MD,MD,O - Foot Patrol,"(39.006675, -77.07532)",2013-08-30 23:39:00,NaN,NaN,NaN,,,single_road,False,False,NaN


In [55]:
traffic_df['location_clean'] = traffic_df['location_clean'].fillna('UNKNOWN')

In [56]:
traffic_df[traffic_df['location_clean'].isnull()]

,seq_id,date_of_stop,time_of_stop,agency,sub_agency,description,location,latitude,longitude,accident,belts,personal_injury,property_damage,fatal,commercial_license,hazmat,commercial_vehicle,alcohol,work_zone,search_conducted,search_disposition,search_outcome,search_reason,search_reason_for_stop,search_type,search_arrest_reason,state,vehicle_type,year,make,model,color,violation_type,charge,article,contributed_to_accident,race,gender,driver_city,driver_state,dl_state,arrest_type,geolocation,stop_timestamp,district_number,direction,exit_ref,road_1,road_2,loc_type,is_landmark,needs_review,location_clean


#### 5.1.6 latitude & longitude

In [57]:
traffic_df['latitude'].isnull().sum()

np.int64(0)

In [58]:
traffic_df['latitude'].dtype

dtype('float64')

In [59]:
traffic_df['latitude'].value_counts()

latitude
0.000000     88414
39.045425      255
39.046277      210
39.058402      192
39.058435      180
             ...  
39.324907        1
38.978000        1
39.024409        1
39.127955        1
39.090807        1
Name: count, Length: 255555, dtype: int64

In [60]:
traffic_df['longitude'].isnull().sum()

np.int64(0)

In [61]:
traffic_df['longitude'].dtype

dtype('float64')

In [62]:
traffic_df['longitude'].value_counts()

longitude
 0.000000     88414
-76.990695      221
-76.990737      213
-77.048067      166
-77.048092      157
              ...  
-76.922412        1
-77.079274        1
-77.230004        1
-77.250094        1
-77.184758        1
Name: count, Length: 279455, dtype: int64

- dataset has no missing values, but lots of wrong values

In [63]:
traffic_df['latitude'] = pd.to_numeric(traffic_df['latitude'], errors='coerce')
traffic_df['longitude'] = pd.to_numeric(traffic_df['longitude'], errors='coerce')

- Remove Zero Coordinates

In [64]:
# Replace (0,0) with NaN
invalid_zero = (traffic_df['latitude'] == 0) & (traffic_df['longitude'] == 0)
traffic_df.loc[invalid_zero, ['latitude', 'longitude']] = None


- Impossible values (Earth-level)

In [65]:
# Remove globally invalid coordinates
traffic_df.loc[
    ~traffic_df['latitude'].between(-90, 90) |
    ~traffic_df['longitude'].between(-180, 180),
    ['latitude', 'longitude']
] = None

In [66]:
traffic_df['latitude'].isna().sum()


np.int64(88414)

In [67]:
traffic_df['longitude'].isna().sum()

np.int64(88414)

- Removed fake coordinates
- Converted wrong data → proper missing values
- Made dataset ready for real geographic analysis

- Out-of-country values (US-level)

In [68]:
#Removing values that are valid globally but not in the US
traffic_df.loc[
    (traffic_df['latitude'] < 24) | (traffic_df['latitude'] > 50) |
    (traffic_df['longitude'] > -65) | (traffic_df['longitude'] < -125),
    ['latitude', 'longitude']
] = None

In [69]:
traffic_df['latitude'].isna().sum()


np.int64(88420)

In [70]:
traffic_df['longitude'].isna().sum()

np.int64(88420)

- Maryland Outlier Detection

In [71]:
# Maryland approximate bounds

traffic_df['is_geo_outlier'] = (
    traffic_df['latitude'].notna() &
    traffic_df['longitude'].notna() &
    (
        ~traffic_df['latitude'].between(37, 40) |
        ~traffic_df['longitude'].between(-79, -75)
    )
)
#checking outliers 
traffic_df['is_geo_outlier'].sum()

np.int64(67)

- Plotting ONLY outliers

In [72]:
outliers = traffic_df[traffic_df['is_geo_outlier']]
outliers['latitude'].value_counts()

latitude
38.870293    40
38.983088     7
38.051859     5
41.543160     4
39.239513     3
40.111822     3
38.254887     2
35.049410     1
41.512073     1
35.942157     1
Name: count, dtype: int64

In [73]:
fig = px.scatter_map(
    outliers,
    lat='latitude',
    lon='longitude',
    zoom=6,
    height=600,
    color_discrete_sequence=['red'],
)

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()


#### 5.1.7 geolocation

- Extract numeric values

In [74]:
# Make sure geolocation is a string first
traffic_df['geolocation'] = traffic_df['geolocation'].astype(str)

# Remove parentheses and split into two columns
traffic_df[['geo_lat', 'geo_lon']] = traffic_df['geolocation'].str.strip('()').str.split(',', expand=True)

# Convert to numeric
traffic_df['geo_lat'] = pd.to_numeric(traffic_df['geo_lat'], errors='coerce')
traffic_df['geo_lon'] = pd.to_numeric(traffic_df['geo_lon'], errors='coerce')

- Remove Zero Coordinates

In [75]:
traffic_df.loc[
    (traffic_df['geo_lat'] == 0) & (traffic_df['geo_lon'] == 0),
    ['geo_lat', 'geo_lon']
] = None

- Impossible values (Earth-level)

In [76]:
# Remove globally invalid coordinates
traffic_df.loc[
    ~traffic_df['geo_lat'].between(-90, 90) |
    ~traffic_df['geo_lon'].between(-180, 180),
    ['geo_lat', 'geo_lon']
] = None

In [77]:
#Removing values that are valid globally but not in the US
traffic_df.loc[
    (traffic_df['geo_lat'] < 24) | (traffic_df['geo_lon'] > 50) |
    (traffic_df['geo_lat'] > -65) | (traffic_df['geo_lon'] < -125),
    ['geo_lat', 'geo_lon']
] = None

In [78]:
traffic_df['geo_lat'].isna().sum()

np.int64(1048575)

In [79]:
traffic_df['geo_lon'].isna().sum()

np.int64(1048575)

In [80]:
missing_latlon = traffic_df.loc[
    (traffic_df['latitude'].isna() | traffic_df['longitude'].isna()) &
    (traffic_df['geo_lat'].notna() & traffic_df['geo_lon'].notna()),
    ['latitude', 'longitude', 'geo_lat', 'geo_lon']
]
print("Rows where latitude/longitude are missing but geo values exist:", len(missing_latlon))


Rows where latitude/longitude are missing but geo values exist: 0


In [81]:
missing_latlon

,latitude,longitude,geo_lat,geo_lon


In [82]:
# Rows where geo_lat/geo_lon are null but latitude/longitude exist
missing_geo = traffic_df.loc[
    (traffic_df['geo_lat'].isna() | traffic_df['geo_lon'].isna()) &
    (traffic_df['latitude'].notna() & traffic_df['longitude'].notna()),
    ['latitude', 'longitude', 'geo_lat', 'geo_lon']
]

print("Rows where geo_lat/geo_lon are missing but lat/lon exist:", len(missing_geo))

Rows where geo_lat/geo_lon are missing but lat/lon exist: 960155


In [83]:
# geo_lat/lon are identical to latitude/longitude, dropping them
traffic_df.drop(columns=['geolocation', 'geo_lat', 'geo_lon'], inplace=True)

#### 5.1.8 traffic_stop table creation

- Step1: Create traffic_stop table

In [84]:
stop_cols = [
    'seq_id', 'stop_timestamp', 'sub_agency', 'district_number',
    'location_clean', 'road_1', 'road_2', 'loc_type', 'is_landmark',
    'latitude', 'longitude'
]
traffic_stop_df = traffic_df[stop_cols].drop_duplicates().reset_index(drop=True)
traffic_stop_df.insert(0, 'stop_id', range(1, len(traffic_stop_df) + 1))

- Step 2: Map stop_id back to original dataframe

In [85]:
traffic_df = traffic_df.merge(traffic_stop_df[['stop_id'] + stop_cols],
                              on=stop_cols, how='left')

In [86]:
traffic_df.head(5)

,seq_id,date_of_stop,time_of_stop,agency,sub_agency,description,location,latitude,longitude,accident,belts,personal_injury,property_damage,fatal,commercial_license,hazmat,commercial_vehicle,alcohol,work_zone,search_conducted,search_disposition,search_outcome,search_reason,search_reason_for_stop,search_type,search_arrest_reason,state,vehicle_type,year,make,model,color,violation_type,charge,article,contributed_to_accident,race,gender,driver_city,driver_state,dl_state,arrest_type,stop_timestamp,district_number,direction,exit_ref,road_1,road_2,loc_type,is_landmark,needs_review,location_clean,is_geo_outlier,stop_id
0,52282e8c-f2e1-4bb5-8509-2d5e4f8da8ca,2023-01-05,23:11:00,MCP,"3rd District, Silver Spring",OPERATING UNREGISTERED MOTOR VEHICLE ON HIGHWAY,BRIGGS CHANEY RD @ COLUMIBA PIKE,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,NaN,Citation,NaN,17-107(a1),NaN,NaN,MD,02 - Automobile,2007.0,CHEV,CRUZ,BLACK,Citation,13-401(b1),Transportation Article,False,WHITE,M,GAITHERSBURG,MD,MD,A - Marked Patrol,2023-01-05 23:11:00,3,NaN,NaN,BRIGGS CHANEY RD,COLUMIBA PIKE,intersection,False,False,BRIGGS CHANEY RD/COLUMIBA PIKE,False,1
1,b66f253b-af29-4bc4-bb73-93755ca2a779,2023-08-31,16:41:00,MCP,"6th District, Gaithersburg / Montgomery Village",DRIVING TO DRIVE MOTOR VEHICLE ON HIGHWAY WITH...,OAKMONT AVE @ GROVEMONT CIR,39.097965,-77.15301,No,No,No,No,No,No,No,No,No,No,No,NaN,Citation,NaN,20-102(a1),NaN,NaN,MD,02 - Automobile,2005.0,FORD,EXPLORER,BLACK,Citation,16-101(a1),Transportation Article,False,HISPANIC,M,GAITHERSBURG,MD,MD,A - Marked Patrol,2023-08-31 16:41:00,6,NaN,NaN,OAKMONT AVE,GROVEMONT CIR,intersection,False,False,OAKMONT AVE/GROVEMONT CIR,False,2
2,b66f253b-af29-4bc4-bb73-93755ca2a779,2023-08-31,16:41:00,MCP,"6th District, Gaithersburg / Montgomery Village",FAILURE TO DISPLAY REGISTRATION CARD UPON DEMA...,OAKMONT AVE @ GROVEMONT CIR,39.097965,-77.15301,No,No,No,No,No,No,No,No,No,No,No,NaN,Citation,NaN,20-102(a1),NaN,NaN,MD,02 - Automobile,2005.0,FORD,EXPLORER,BLACK,Citation,13-409(b),Transportation Article,False,HISPANIC,M,GAITHERSBURG,MD,MD,A - Marked Patrol,2023-08-31 16:41:00,6,NaN,NaN,OAKMONT AVE,GROVEMONT CIR,intersection,False,False,OAKMONT AVE/GROVEMONT CIR,False,2
3,b66f253b-af29-4bc4-bb73-93755ca2a779,2023-08-31,16:41:00,MCP,"6th District, Gaithersburg / Montgomery Village",DRIVER OF MOTOR VEHICLE FOLLOWING VEHICLE CLOS...,OAKMONT AVE @ GROVEMONT CIR,39.097965,-77.15301,No,No,No,No,No,No,No,No,No,No,No,NaN,Citation,NaN,20-102(a1),NaN,NaN,MD,02 - Automobile,2005.0,FORD,EXPLORER,BLACK,Citation,21-310(a),Transportation Article,False,HISPANIC,M,GAITHERSBURG,MD,MD,A - Marked Patrol,2023-08-31 16:41:00,6,NaN,NaN,OAKMONT AVE,GROVEMONT CIR,intersection,False,False,OAKMONT AVE/GROVEMONT CIR,False,2
4,b66f253b-af29-4bc4-bb73-93755ca2a779,2023-08-31,16:41:00,MCP,"6th District, Gaithersburg / Montgomery Village",FAILURE TO CONTROL VEH. SPEED ON HWY. TO AVOID...,OAKMONT AVE @ GROVEMONT CIR,39.097965,-77.15301,No,No,No,No,No,No,No,No,No,No,No,NaN,Citation,NaN,20-102(a1),NaN,NaN,MD,02 - Automobile,2005.0,FORD,EXPLORER,BLACK,Citation,21-801(b),Transportation Article,False,HISPANIC,M,GAITHERSBURG,MD,MD,A - Marked Patrol,2023-08-31 16:41:00,6,NaN,NaN,OAKMONT AVE,GROVEMONT CIR,intersection,False,False,OAKMONT AVE/GROVEMONT CIR,False,2


In [87]:
# ============================================================
# TRAFFIC STOP — DDL PREPARATION
# ============================================================

print(f"Total rows:       {traffic_stop_df.shape[0]:,}")
print(f"Unique stop_ids:  {traffic_stop_df['stop_id'].nunique():,}")
print(f"NaN stop_ids:     {traffic_stop_df['stop_id'].isna().sum():,}")
print(f"Is stop_id unique:{traffic_stop_df['stop_id'].nunique() == traffic_stop_df.shape[0]}")

# Check varchar lengths
str_cols = ['sub_agency', 'location_clean', 'road_1', 'road_2', 'loc_type']
print(f"\nMax lengths:")
for col in str_cols:
    max_len = traffic_stop_df[col].str.len().max()
    print(f"  {col:20} max length: {max_len}")

# Check dtypes
print(f"\nDtypes:")
print(traffic_stop_df.dtypes)

# Check NaN counts
print(f"\nNaN counts:")
print(traffic_stop_df.isna().sum())

Total rows:       568,317
Unique stop_ids:  568,317
NaN stop_ids:     0
Is stop_id unique:True

Max lengths:
  sub_agency           max length: 47
  location_clean       max length: 40
  road_1               max length: 41
  road_2               max length: 39
  loc_type             max length: 12

Dtypes:
stop_id                     int64
seq_id                     object
stop_timestamp     datetime64[ns]
sub_agency                 object
district_number            object
location_clean             object
road_1                     object
road_2                     object
loc_type                   object
is_landmark                  bool
latitude                  float64
longitude                 float64
dtype: object

NaN counts:
stop_id                0
seq_id                 0
stop_timestamp         0
sub_agency             0
district_number    52459
location_clean         0
road_1                 0
road_2                 0
loc_type               0
is_landmark            0
latitud

- district_number is string not numeric:

In [88]:
# Fix district_number — convert string to Int64 (nullable)
traffic_stop_df['district_number'] = (
    traffic_stop_df['district_number']
    .astype(float)      # handles NaN first
    .astype('Int64')    # nullable integer
)

print(traffic_stop_df['district_number'].dtype)
print(traffic_stop_df['district_number'].value_counts())
print(f"NaN: {traffic_stop_df['district_number'].isna().sum():,}")

Int64
district_number
2    104907
4    101866
3     93740
5     81143
1     70103
6     64099
Name: count, dtype: Int64
NaN: 52,459


## 5.2 Incident_safety 

In [89]:
incident_cols = [
    'accident', 'belts', 'personal_injury', 
    'property_damage', 'fatal', 'contributed_to_accident'
]

traffic_df[incident_cols].isna().sum()


accident                   0
belts                      0
personal_injury            0
property_damage            0
fatal                      0
contributed_to_accident    0
dtype: int64

- No missing values found

In [90]:
print(traffic_df[incident_cols].dtypes)

accident                   object
belts                      object
personal_injury            object
property_damage            object
fatal                      object
contributed_to_accident      bool
dtype: object


In [91]:
for col in incident_cols:
    print(f"{traffic_df[col].value_counts()}")

accident
No     998242
Yes     50333
Name: count, dtype: int64
belts
No     1007996
Yes      40579
Name: count, dtype: int64
personal_injury
No     1025976
Yes      22599
Name: count, dtype: int64
property_damage
No     1009269
Yes      39306
Name: count, dtype: int64
fatal
No     1048174
Yes        401
Name: count, dtype: int64
contributed_to_accident
False    998242
True      50333
Name: count, dtype: int64


- All coulmns are boolean datatypes

##### 5.2.1 Standardize categorical columns

In [92]:
for col in incident_cols:
    traffic_df[col] = traffic_df[col].map({
        'Yes': 1,
        'No': 0,
        True: 1,
        False: 0
    }).astype(int)

- check unique values

In [93]:
for col in incident_cols:
    print(col, traffic_df[col].unique())

accident [0 1]
belts [0 1]
personal_injury [0 1]
property_damage [0 1]
fatal [0 1]
contributed_to_accident [0 1]


- Step1: Create incident_safety table

In [94]:
incident_safety_df = traffic_df[['stop_id'] + incident_cols].copy()

In [95]:
incident_safety_df.head(10)

,stop_id,accident,belts,personal_injury,property_damage,fatal,contributed_to_accident
0,1,0,0,0,0,0,0
1,2,0,0,0,0,0,0
2,2,0,0,0,0,0,0
3,2,0,0,0,0,0,0
4,2,0,0,0,0,0,0
5,3,0,0,0,0,0,0
6,2,0,0,0,0,0,0
7,2,0,0,0,0,0,0
8,2,0,0,0,0,0,0
9,2,0,0,0,0,0,0


- Create Surrogate Primary key

In [96]:
incident_safety_df = incident_safety_df.reset_index(drop=True)
incident_safety_df.insert(0, 'incident_id', range(1, len(incident_safety_df) + 1))

In [97]:
incident_safety_df.head(5)

,incident_id,stop_id,accident,belts,personal_injury,property_damage,fatal,contributed_to_accident
0,1,1,0,0,0,0,0,0
1,2,2,0,0,0,0,0,0
2,3,2,0,0,0,0,0,0
3,4,2,0,0,0,0,0,0
4,5,2,0,0,0,0,0,0


In [98]:
# ============================================================
# INCIDENT SAFETY — DEDUPLICATION
# ============================================================

# Keep row with accident=1 if any row has it
incident_safety_df = (
    incident_safety_df
    .sort_values(
        ['accident', 'contributed_to_accident'],
        ascending=False  # 1 comes before 0
    )
    .drop_duplicates(
        subset=['stop_id'],
        keep='first'     # keep accident=1 row
    )
    .reset_index(drop=True)
)

# Validation
print(f"Final shape:     {incident_safety_df.shape}")
print(f"Unique stop_ids: {incident_safety_df['stop_id'].nunique():,}")
print(f"\nAccident counts:")
print(incident_safety_df['accident'].value_counts())


Final shape:     (568317, 8)
Unique stop_ids: 568,317

Accident counts:
accident
0    546608
1     21709
Name: count, dtype: int64


## 5.3 Driver_Vehicle 

- driver_vehicle includes these columns:
    + driver_id (new surrogate key)
    + stop_id (FK)
    + race
    + gender
    + driver_city
    + driver_state
    + dl_state
    + commercial_license
    + hazmat
    + commercial_vehicle
    + alcohol
    + work_zone
    + vehicle_type
    + year
    + make
    + model
    + color

- Step 2: Clean boolean columns

In [99]:
bool_cols = [
    'commercial_license', 'hazmat', 'commercial_vehicle', 'alcohol', 'work_zone'
]

In [100]:
traffic_df[bool_cols].isna().sum()


commercial_license    0
hazmat                0
commercial_vehicle    0
alcohol               0
work_zone             0
dtype: int64

- No missing values found

In [101]:
print(traffic_df[bool_cols].dtypes)

commercial_license    object
hazmat                object
commercial_vehicle    object
alcohol               object
work_zone             object
dtype: object


In [102]:
for col in bool_cols:
    print(f"{traffic_df[col].value_counts()}")

commercial_license
No     1016347
Yes      32228
Name: count, dtype: int64
hazmat
No     1048512
Yes         63
Name: count, dtype: int64
commercial_vehicle
No     1045673
Yes       2902
Name: count, dtype: int64
alcohol
No     1046088
Yes       2487
Name: count, dtype: int64
work_zone
No     1048247
Yes        328
Name: count, dtype: int64


In [103]:
for col in bool_cols:
    traffic_df[col] = traffic_df[col].map({'Yes': 1, 'No': 0, True: 1, False: 0}).fillna(0).astype(int)

- Converts all Yes/No/True/False → 0/1 integers

- Check unique values

In [104]:
for col in bool_cols:
    print(col, traffic_df[col].unique())

commercial_license [0 1]
hazmat [0 1]
commercial_vehicle [0 1]
alcohol [0 1]
work_zone [0 1]


- Step 3: Standardize categorical/text columns

In [105]:
cat_cols = ['race', 'gender', 'driver_state', 'dl_state', 'vehicle_type', 'make', 'model', 'color', 'driver_city']


In [106]:

for col in cat_cols:
    traffic_df[col] = traffic_df[col].str.upper().str.strip()

#### 5.3.1 race

In [107]:
traffic_df['race'].value_counts()

race
WHITE              348788
BLACK              329224
HISPANIC           255721
OTHER               61204
ASIAN               51890
NATIVE AMERICAN      1748
Name: count, dtype: int64

In [108]:
traffic_df['race'].isna().sum()

np.int64(0)

- **Values:**
  - WHITE
  - BLACK
  - HISPANIC
  - ASIAN
  - OTHER
  - NATIVE AMERICAN
- **Cleaning Performed:**
  - Standardized to uppercase
  - Trimmed whitespace
  - No missing values found
- **Notes:**
  - No imputation performed
  - Rare categories retained to preserve data integrity

#### 5.3.2 gender

In [109]:
traffic_df['gender'].isna().sum()

np.int64(0)

In [110]:
traffic_df['gender'].value_counts()

gender
M    728722
F    318812
U      1041
Name: count, dtype: int64

- **Values:**
  - M → Male
  - F → Female
  - U → Unknown
- **Cleaning Performed:**
  - No missing values found
  - Standardized to uppercase
  - Trimmed whitespace
- **Notes:**
  - Unknown category retained
  - No imputation performed

#### 5.3.3 driver_state

In [111]:
traffic_df['driver_state'].isna().sum()

np.int64(8)

In [112]:
traffic_df['driver_state'].value_counts()

driver_state
MD    951915
DC     34755
VA     31772
PA      5176
FL      3060
       ...  
NB         3
US         3
GU         2
BC         1
SK         1
Name: count, Length: 64, dtype: int64

In [113]:
traffic_df['driver_state'] = traffic_df['driver_state'].fillna('UNKNOWN')

In [114]:
traffic_df['driver_state'] = traffic_df['driver_state'].replace('XX', 'UNKNOWN')

In [115]:
valid_states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
    'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY',
    'DC','GU','PR','VI','AS'
]

traffic_df['driver_state'] = traffic_df['driver_state'].apply(
    lambda x: x if x in valid_states else 'UNKNOWN'
)

In [116]:
traffic_df['driver_state'].value_counts()

driver_state
MD         951915
DC          34755
VA          31772
PA           5176
FL           3060
NY           2741
WV           2345
NC           2151
CA           1421
TX           1362
NJ           1355
GA           1214
MA            930
DE            875
UNKNOWN       735
OH            725
SC            552
IL            445
CT            400
WA            400
MI            383
TN            351
IN            310
CO            281
AZ            271
AL            236
MO            232
LA            218
MS            149
NV            140
NM            132
OK            124
KY            122
MN            120
WI            118
ME            117
UT            108
RI            102
NH             87
ND             78
KS             77
OR             68
AK             50
HI             46
NE             46
AR             46
VT             45
IA             43
MT             41
ID             35
SD             22
WY             20
PR             18
VI              8
GU             

- **Cleaning Performed:**
  - Converted all values to uppercase
  - Trimmed whitespace
  - Replaced invalid code "XX" with "UNKNOWN"
  - Filled missing values with "UNKNOWN"
  - (Optional) Filtered invalid codes to "UNKNOWN"

- **Top Values:**
  - MD (majority)
  - DC, VA (neighboring regions)

- **Notes:**
  - No inference performed
  - Unknown values explicitly labeled

#### 5.3.4 dl_state

In [117]:
traffic_df['dl_state'].isna().sum()

np.int64(819)

In [118]:
traffic_df['dl_state'].value_counts()

dl_state
MD    907619
DC     34653
VA     33882
XX     23035
PA      6270
       ...  
GU         5
AS         3
NS         2
PQ         1
NF         1
Name: count, Length: 70, dtype: int64

In [119]:
traffic_df['dl_state'] = traffic_df['dl_state'].fillna('UNKNOWN')
traffic_df['dl_state'] = traffic_df['dl_state'].replace('XX', 'UNKNOWN')

In [120]:
valid_states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
    'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY',
    'DC','GU','PR','VI','AS'
]

traffic_df['dl_state'] = traffic_df['dl_state'].apply(
    lambda x: x if x in valid_states else 'UNKNOWN'
)

In [121]:
traffic_df['dl_state'].value_counts()

dl_state
MD         907619
DC          34653
VA          33882
UNKNOWN     24801
PA           6270
FL           5542
NY           4436
NC           3413
CA           3009
TX           2691
WV           2422
GA           2216
NJ           2059
MA           1548
OH           1141
DE           1096
SC           1034
IL            892
WA            790
AZ            715
MI            703
TN            662
CT            653
CO            632
IN            466
MO            402
LA            399
AL            366
NV            248
MS            244
ME            232
WI            231
MN            230
KY            230
NM            223
UT            212
OK            210
VI            210
RI            204
OR            162
KS            157
IA            153
NH            147
HI            128
AK            124
AR            106
MT            104
ID            103
ND            101
NE             82
PR             77
VT             71
SD             39
WY             27
GU              5
A

- **Cleaning Performed:**
  - Converted to uppercase
  - Trimmed whitespace
  - Replaced "XX" with "UNKNOWN"
  - Filled missing values with "UNKNOWN"
  - (Optional) Filtered invalid codes to "UNKNOWN"
  - Includes US states and territories
  - Unknown/invalid values explicitly labeled

#### 5.3.5 vehicle_type

In [122]:
traffic_df['vehicle_type'].isna().sum()

np.int64(0)

In [123]:
traffic_df['vehicle_type'].value_counts()

vehicle_type
02 - AUTOMOBILE              903541
05 - LIGHT DUTY TRUCK         66479
28 - OTHER                    27777
03 - STATION WAGON            17174
01 - MOTORCYCLE               10792
06 - HEAVY DUTY TRUCK          8210
29 - UNKNOWN                   5990
08 - RECREATIONAL VEHICLE      3170
19 - MOPED                     1606
07 - TRUCK/ROAD TRACTOR         981
20 - COMMERCIAL RIG             644
04 - LIMOUSINE                  609
25 - UTILITY TRAILER            499
10 - TRANSIT BUS                432
12 - SCHOOL BUS                 284
27 - FARM EQUIPMENT             110
26 - BOAT TRAILER                57
09 - FARM VEHICLE                47
11 - CROSS COUNTRY BUS           39
21 - TANDEM TRAILER              22
23 - TRAVEL/HOME TRAILER         20
22 - MOBILE HOME                 18
18 - POLICE(NON-EMERG)           15
13 - AMBULANCE(EMERG)            14
15 - FIRE(EMERG)                 12
24 - CAMPER                      11
14 - AMBULANCE(NON-EMERG)         9
16 - FIRE(NON-E

In [124]:
# Replace different dashes with standard "-" and fix spacing
traffic_df['vehicle_type_clean'] = (
    traffic_df['vehicle_type']
    .str.replace(r'[–—]', '-', regex=True)   # replace en-dash/em-dash with -
    .str.replace(r'\s*-\s*', ' - ', regex=True)  # ensure single spaces around -
    .str.upper()  # optional: uppercase everything for consistency
    .str.strip()
)

In [125]:
# Split only on the first " - " and allow missing
split_cols = traffic_df['vehicle_type_clean'].str.split(' - ', n=1, expand=True)

# Assign columns safely
traffic_df['vehicle_code'] = split_cols[0]
traffic_df['vehicle_category'] = split_cols[1]  # will be NaN if missing

In [126]:
traffic_df['vehicle_type']=traffic_df['vehicle_type_clean']
traffic_df.drop(columns='vehicle_type_clean',inplace=True)

- Replaced original messy column with cleaned version
- Removed temporary column → keeps dataset clean

#### 5.3.6 year

In [127]:
traffic_df['year'].value_counts()

year
2006.0    57718
2007.0    57621
2005.0    55261
2012.0    55110
2013.0    53935
          ...  
5.0           1
2483.0        1
9.0           1
6247.0        1
4706.0        1
Name: count, Length: 285, dtype: int64

In [128]:
traffic_df['year'].dtype

dtype('float64')

In [129]:
traffic_df['year'].isnull().sum()

np.int64(5700)

- check invalid years

In [130]:
(~traffic_df['year'].between(1960, 2025)).sum()

np.int64(7375)

- 5700 were already missing
- 1675 were invalid years 

Step 1: Inspect invalid values

In [131]:
impossible_years = traffic_df.loc[
    ~traffic_df['year'].between(1960, 2025, inclusive='both'),
    'year'
]
impossible_years.value_counts()

year
0.0       829
1900.0    106
9999.0     43
2912.0     23
2103.0     21
         ... 
5.0         1
2483.0      1
9.0         1
6247.0      1
4706.0      1
Name: count, Length: 221, dtype: int64

Step 2: Clean (remove invalid)

In [132]:
traffic_df['year'] = traffic_df['year'].where(
    traffic_df['year'].between(1960, 2025)
)

Step 3: Convert to integer (nullable)

In [133]:
traffic_df['year'] = traffic_df['year'].astype('Int64')

In [134]:
traffic_df['year'].isnull().sum()

np.int64(7375)

- now, Total NaN = 5700 + 1675 = 7375 ✅
- Invalid values are now treated as missing

In [135]:
traffic_df['year'].dtype

Int64Dtype()

- Removed invalid years (<1960 or >2025)
- Converted invalid values to NULL
- Total missing values increased from 5700 → 7375
- Converted column to nullable integer (Int64)

#### 5.3.7 make

In [136]:
traffic_df['make'].isna().sum()

np.int64(25)

In [137]:
traffic_df['make'].value_counts().sample(20,random_state=30)

make
HUYN 2D            1
CADILLA           12
CHE V              2
MITSHUBISHI       23
VOLO              13
MERCEDES BEMZ      2
F550               1
ACUA               8
VOLCO              1
SATURNQ            2
FRAIGHT LINER      3
MITZUBISHI        26
MIT               45
CHWVY              3
VOLKE             17
MERCEDES-BENZ    206
HONIDA             5
LOAD               7
TOYOTA`            1
MITSUUBISHI        1
Name: count, dtype: int64

- Fix obvious typos (manual mapping)

- Step 1 — Define the full map 

In [138]:
make_map = {
    'ACUR' : 'ACURA',
    'ACUA': 'ACURA',
    'ACURA TL' : 'ACURA',

    'X5BMW' : 'BMW',

    'BUIC'   : 'BUICK',
    'BIUICK' : 'BUICK',

    'CHEV': 'CHEVROLET',
    'CHE V': 'CHEVROLET',
    'CHRV' : 'CHEVROLET',
    'CHWVY': 'CHEVROLET',
    'CHEVY'    : 'CHEVROLET',
    'CHEVY SUB' : 'CHEVROLET',
    'CHEVRIOLET' : 'CHEVROLET',
    'CHEVORLET'   : 'CHEVROLET',
    'CHEVEROLET' : 'CHEVROLET',
    
    'CADI'      : 'CADILLAC',
    'CADILLA': 'CADILLAC',
    'CADILACK' : 'CADILLAC',
    'CADILAC'    : 'CADILLAC',

    'CHRY'      : 'CHRYSLER',
    'CHREYSLER': 'CHRYSLER',
    'CHRYS'    : 'CHRYSLER',
    'CHRYSTLER': 'CHRYSLER',

    'FRAIGHT LINER': 'FREIGHTLINER',
    'FRHT'       : 'FREIGHTLINER',     # truck abbreviation

    'FIAT' : 'FIAT',            # valid make

    'MUSTANG' : 'FORD',

    'GRUMAN' : 'GRUMMAN',

    'GEO'    : 'GEO',

    'HOND' : 'HONDA',
    'HONDA TRUCK' : 'HONDA',
    'HONIDA': 'HONDA',
    'HONDA 2S': 'HONDA',
    'HONDA2012' : 'HONDA',

    'HYUN'     : 'HYUNDAI',
    'HYUN'     : 'HYUNDAI',
    'HYUNUDAI' : 'HYUNDAI',
    'HTUN'     : 'HYUNDAI',
    'HUYN 2D'  : 'HYUNDAI',
    'HYUNDA'   : 'HYUNDAI',
    'HYUNDAIN' : 'HYUNDAI',
    'HYU NDAI' : 'HYUNDAI',
    'HYUND'    : 'HYUNDAI',
    'HYUNDIA'  : 'HYUNDAI',

    'HINO'       : 'HINO',

    'HUMMER'     : 'HUMMER',           # valid make (GM sub-brand)

    'ISUZU'         : 'ISUZU',
    'ISUZ'        : 'ISUZU',
    'ISU'         : 'ISUZU',

    'INFI'      : 'INFINITI',
    'INFINITY'      : 'INFINITI',
    'INFIN'      : 'INFINITI',

    'INTERNATIONAL': 'INTERNATIONAL',
    'INTL'        : 'INTERNATIONAL',
  
    'DODG'      : 'DODGE', # was DODGE/JEEP before
    'JEEK'       : 'JEEP',   

    'PACE' : 'JAGUAR',
    'PACE SPORT' : 'JAGUAR',
    'JAGU'        : 'JAGUAR',

    'KAWASAKI'    : 'KAWASAKI',        # motorcycle, valid
    'KENWORTH'   : 'KENWORTH',

    'LAND RANGER'  : 'LAND ROVER',
    'LAN ROVER'    : 'LAND ROVER',
    'LANDROVER'    : 'LAND ROVER',
    'LAND ROVWR'   : 'LAND ROVER',
    'RANGE ROVER'  : 'LAND ROVER',
    'LNDR'         : 'LAND ROVER',
    
    'LEXUES' : 'LEXUS',
    ';EXUS' : 'LEXUS',
    'LEXS'  : 'LEXUS',
    'LEXU'  : 'LEXUS',
    'LEX'   : 'LEXUS',

    'LINC'          : 'LINCOLN',
    
    'MERZ BENZ'  : 'MERCEDES-BENZ',
    'MERZ'     : 'MERCEDES-BENZ',
    'MERCEDES'   : 'MERCEDES-BENZ',
    'MERCEDES BEMZ': 'MERCEDES-BENZ',
    'MERCEDEZ'      : 'MERCEDES-BENZ',
    'MERCEDES BENZ' : 'MERCEDES-BENZ',
    'BENZ' : 'MERCEDES-BENZ',
    'NERCEDEZ' : 'MERCEDES-BENZ',
    'MERCEDSES' : 'MERCEDES-BENZ',

    'MERCURY': 'MERCURY',         # Ford sub-brand, valid make
    'MERC'          : 'MERCURY',

    'MITSBISHI' : 'MITSUBISHI',
    'MITZUBISHI': 'MITSUBISHI',
    'MITSHUBISHI': 'MITSUBISHI',
    'MITSUUBISHI': 'MITSUBISHI',
    'MIT'        : 'MITSUBISHI',
    'MITS'       : 'MITSUBISHI',
    'MITZ'       : 'MITSUBISHI',
    'MITSU'      : 'MITSUBISHI',

    'MASD' : 'MAZDA',
    'MAZD'      : 'MAZDA',

    'MINI-COOPER' : 'MINI',
    'MNNI'       : 'MINI',

    'MACK'        : 'MACK',            # truck brand, valid

    'NISS'        : 'NISSAN',
    'NISSVAL2013' : 'NISSAN',
    'NISSVAL2011' : 'NISSAN',
    'N5SSAN'      : 'NISSAN',
    'NISSIAN'     : 'NISSAN',

    'OLDS'  : 'OLDSMOBILE',

    'PERERBUILT' : 'PETERBILT',

    'GT TEMPEST' : 'PONTIAC',
    'PONT'       : 'PONTIAC',

    'PORCSCHE' : 'PORSCHE',
    'PORS'        : 'PORSCHE',

    'PLYMOUTH'    : 'PLYMOUTH',        # valid historical make
    'PLYM'       : 'PLYMOUTH',

    'SATURNQ' : 'SATURN',
    'SATU'    : 'SATURN',
    'STRN'    : 'SATURN',

    'SSR MOTORSPORT' : 'SSR MOTORSPORTS',

    'SAAB'          : 'SAAB',

    'SUSUBARU' : 'SUBARU',
    'SUBA'     : 'SUBARU',
    'SUB'      : 'SUBARU',
    'SUBURU'     : 'SUBARU',

    'ZUZUKI': 'SUZUKI',
    'SUZUK' : 'SUZUKI',
    'SUZI'  : 'SUZUKI',
    'SUZU'  : 'SUZUKI',

    'SCION' : 'SCION',           # Toyota's sub-brand, valid make
    'SCIO'  : 'SCION',

    'TAOTAO'     : 'TAOTAO',

    'TOYTA'       : 'TOYOTA',
    'LANDCRUISER' : 'TOYOTA',
    '4 RUNNERTOYOTA' : 'TOYOTA',
    'TOYO'   : 'TOYOTA',
    '1=TOYT' : 'TOYOTA',
    'OIYOTA' : 'TOYOTA',
    'SOLARA' : 'TOYOTA',            
    'TOYT'   : 'TOYOTA',
    'TOYTS'  : 'TOYOTA',
    'TOYOYT' : 'TOYOTA',
    'TOYOT'  : 'TOYOTA',
    'TOY'    : 'TOYOTA',

    'TESAL' : 'TESLA',
    'TESL'  : 'TESLA',


    'VOLKE'     : 'VOLKSWAGEN',
    'VOLK'      : 'VOLKSWAGEN',
    'VOLKS'     : 'VOLKSWAGEN',
    'VOLKSWAGON': 'VOLKSWAGEN',
    'VW'        : 'VOLKSWAGEN',
    
    'VOLO' : 'VOLVO',
    'VOLCO': 'VOLVO',  
    'VOLV' : 'VOLVO',

    'YAMAHA': 'YAMAHA', 

    'XXXX' : 'UNKNOWN',
    'NONE' : 'UNKNOWN',
    'WELL' : 'UNKNOWN',
    'VALL' : 'UNKNOWN',
    'SUPM' : 'UNKNOWN',
    'C'    : 'UNKNOWN',
    'DODGE/JEEP' : 'UNKNOWN',
    'NINGBO' : 'UNKNOWN',

# --- ROUND 6 ADDITIONS ---
'YAMA'            : 'YAMAHA',
'MASERATI'        : 'MASERATI',
'HYUNDI'          : 'HYUNDAI',
'TOYTOA'          : 'TOYOTA',
'PTRB'            : 'PETERBILT',       # truck brand
'TAO TAO'         : 'TAOTAO',
'HARLEY DAVIDSON' : 'HARLEY-DAVIDSON',
'KAWK'            : 'KAWASAKI',
'CRYSLER'         : 'CHRYSLER',
'MAZ'             : 'MAZDA',
'SMART'           : 'SMART',           # valid make (Mercedes sub-brand)
'KW'              : 'KENWORTH',
'CAD'             : 'CADILLAC',
'MAZADA'          : 'MAZDA',
'HUMM'            : 'HUMMER',
'THOMAS'          : 'THOMAS',          # school bus manufacturer, valid
'SUBU'            : 'SUBARU',
'NISSA'           : 'NISSAN',
'JAG'             : 'JAGUAR',
'HYUNDAY'         : 'HYUNDAI',
'MINI COOPER'     : 'MINI',
'HYUDAI'          : 'HYUNDAI',
'TAIZHOU'         : 'TAIZHOU',         # Chinese scooter brand, valid
'RANG'            : 'LAND ROVER',
'FREI'            : 'FREIGHTLINER',
'IZUZU'           : 'ISUZU',
'INFINTI'         : 'INFINITI',
'TOYOYA'          : 'TOYOTA',
'PETE'            : 'PETERBILT',
'HINDA'           : 'HONDA',
'INF'             : 'INFINITI',
'SATR'            : 'SATURN',
'HARLEY'          : 'HARLEY-DAVIDSON',
'TOTY'            : 'TOYOTA',
'PETERBUILT'      : 'PETERBILT',
'MERCEDEZ BENZ'   : 'MERCEDES-BENZ',
'PORCHE'          : 'PORSCHE',
'VOLKS WAGON'     : 'VOLKSWAGEN',
'TOTOTA'          : 'TOYOTA',
'VOLKWAGON'       : 'VOLKSWAGEN',

# --- ROUND 7 ADDITIONS ---
'SAA'      : 'SAAB',
'HON'      : 'HONDA',
'HYNDAI'   : 'HYUNDAI',
'MADZA'    : 'MAZDA',
'MASE'     : 'MASERATI',
'INFINIT'  : 'INFINITI',
'LAND'     : 'LAND ROVER',
'HONA'     : 'HONDA',
'CHYSLER'  : 'CHRYSLER',
'HUNDAI'   : 'HYUNDAI',
'STERLING' : 'STERLING',    # valid truck brand
'TOYOA'    : 'TOYOTA',
'KENW'     : 'KENWORTH',
'CHEVE'    : 'CHEVROLET',
'NISAN'    : 'NISSAN',
'CHEVEY'   : 'CHEVROLET',
'ACCURA'   : 'ACURA',
'GILL'     : 'UNKNOWN',     # unclear — could be anything
'TSMR'     : 'UNKNOWN',     # unclear
'UD'       : 'UD',          # Japanese truck brand, valid

# --- ROUND 8 ADDITIONS (50+ group) ---
'HYANDAI'    : 'HYUNDAI',
'HUYNDAI'    : 'HYUNDAI',
'DUCATI'     : 'DUCATI',          # motorcycle, valid
'CHEVERLOT'  : 'CHEVROLET',
'VESPA'      : 'VESPA',           # scooter, valid
'CHYRSLER'   : 'CHRYSLER',
'SAT'        : 'SATURN',
'FREIGHT'    : 'FREIGHTLINER',
'UNK'        : 'UNKNOWN',
'HUYN'       : 'HYUNDAI',
'ACRUA'      : 'ACURA',
'MER'        : 'MERCEDES-BENZ',
'BWM'        : 'BMW',
'KYMCO'      : 'KYMCO',           # scooter brand, valid
'VOLSWAGON'  : 'VOLKSWAGEN',
'MERZEDES'   : 'MERCEDES-BENZ',
'MERC BENZ'  : 'MERCEDES-BENZ',
'NEW FLYER'  : 'NEW FLYER',       # bus manufacturer, valid
'SMRT'       : 'SMART',
'HNDA'       : 'HONDA',
'RANGE'      : 'LAND ROVER',
'GILLIG'     : 'GILLIG',          # bus manufacturer, valid
'TOYTOTA'    : 'TOYOTA',
'MITTS'      : 'MITSUBISHI',
'MISTUBISHI' : 'MITSUBISHI',
'VOLKWAGEN'  : 'VOLKSWAGEN',
'PORSHE'     : 'PORSCHE',
'HYND'       : 'HYUNDAI',
'CEHVY'      : 'CHEVROLET',
'SUZ'        : 'SUZUKI',
'CRYSTLER'   : 'CHRYSLER',
'SUBURA'     : 'SUBARU',
'INFINI'     : 'INFINITI',
'INT'        : 'INTERNATIONAL',
'INFINITE'   : 'INFINITI',
'CARR'       : 'UNKNOWN',         # unclear
'ALFA ROMEO' : 'ALFA ROMEO',      # valid make
'INTE'       : 'INTERNATIONAL',
'CHRYLSER'   : 'CHRYSLER',
'DUCA'       : 'DUCATI',
'INIFINITI'  : 'INFINITI',

# --- ROUND 9 ADDITIONS (20-49 group) ---
'CADDILAC'        : 'CADILLAC',
'HODA'            : 'HONDA',
'THOM'            : 'THOMAS',
'BENTLEY'         : 'BENTLEY',        # valid make
'HYUANDI'         : 'HYUNDAI',
'CRYS'            : 'CHRYSLER',
'FOR'             : 'FORD',
'FLY WING'        : 'UNKNOWN',        # obscure Chinese brand
'ORION'           : 'ORION',          # bus manufacturer, valid
'VOLKSWAGGON'     : 'VOLKSWAGEN',
'GENESIS'         : 'GENESIS',        # Hyundai sub-brand, valid
'LEXIS'           : 'LEXUS',
'TOMOS'           : 'TOMOS',          # moped brand, valid
'TRIUMPH'         : 'TRIUMPH',        # motorcycle, valid
'ZHEJIANG'        : 'UNKNOWN',        # generic Chinese manufacturer
'MITSUBUSHI'      : 'MITSUBISHI',
'DAEWOO'          : 'DAEWOO',         # valid make
'MAST'            : 'UNKNOWN',
'WHITE'           : 'WHITE',          # truck brand, valid
'SION'            : 'SCION',
'MECURY'          : 'MERCURY',
'DOGE'            : 'DODGE',
'TOYOYTA'         : 'TOYOTA',
'HD'              : 'HARLEY-DAVIDSON',
'STER'            : 'STERLING',
'CHRISLER'        : 'CHRYSLER',
'COOPER'          : 'MINI',
'TYOTA'           : 'TOYOTA',
'ZHEJIANG JIAJUE' : 'UNKNOWN',
'CHEVROLETE'      : 'CHEVROLET',
'HYU'             : 'HYUNDAI',
'4S'              : 'UNKNOWN',
'XX'              : 'UNKNOWN',
'SUZIKI'          : 'SUZUKI',
'MB'              : 'MERCEDES-BENZ',
'VOLKSWAGAN'      : 'VOLKSWAGEN',
'INIFI'           : 'INFINITI',
'ALFA'            : 'ALFA ROMEO',
'MINNI'           : 'MINI',
'JONWAY'          : 'UNKNOWN',        # Chinese scooter brand
'MITSIBISHI'      : 'MITSUBISHI',
'HIONDA'          : 'HONDA',
'INFINTY'         : 'INFINITI',
'HOMDA'           : 'HONDA',
'HODNA'           : 'HONDA',
'HYN'             : 'HYUNDAI',
'VOLKSW'          : 'VOLKSWAGEN',
'SCOOTER'         : 'UNKNOWN',
'CHECY'           : 'CHEVROLET',
'HUNDAY'          : 'HYUNDAI',
'TOOTA'           : 'TOYOTA',
'INTER'           : 'INTERNATIONAL',
'SUNNY'           : 'UNKNOWN',
'HYUNAI'          : 'HYUNDAI',
'CHRSYLER'        : 'CHRYSLER',
'MAZA'            : 'MAZDA',
'NISSAM'          : 'NISSAN',
'SUBAR'           : 'SUBARU',
'MITIS'           : 'MITSUBISHI',
'CHEROLET'        : 'CHEVROLET',
'MIFU'            : 'UNKNOWN',
'CADILLIAC'       : 'CADILLAC',
'WEST'            : 'UNKNOWN',
'BUELL'           : 'BUELL',          # motorcycle brand, valid
'KAW'             : 'KAWASAKI',
'FOED'            : 'FORD',
'FERRARI'         : 'FERRARI',        # valid make
'SUSUKI'          : 'SUZUKI',
'VOLV0'           : 'VOLVO',          # zero not O
'YONGFU'          : 'UNKNOWN',
'KAWASKI'         : 'KAWASAKI',
'ORD'             : 'FORD',
'EAGLE'           : 'EAGLE',          # valid historical make
'TAIWAN GOLDEN B' : 'UNKNOWN',
'VOLKSWA'         : 'VOLKSWAGEN',
'AMC'             : 'AMC',            # valid historical make
'MITTSUBISHI'     : 'MITSUBISHI',
'MITSUBSHI'       : 'MITSUBISHI',
'PREM'            : 'UNKNOWN',
'VOLKWAG'         : 'VOLKSWAGEN',
'MERCADES'        : 'MERCEDES-BENZ',
'TOTOYA'          : 'TOYOTA',
'GENS'            : 'GENESIS',
'VOKSWAGON'       : 'VOLKSWAGEN',
'TOYORA'          : 'TOYOTA',
'MAD'             : 'MAZDA',
'VOLS'            : 'VOLVO',
'VOLTSWAGON'      : 'VOLKSWAGEN',
'VOLSWAGEN'       : 'VOLKSWAGEN',
'DIDGE'           : 'DODGE',
'DONGFANG'        : 'UNKNOWN',
'SSR'             : 'SSR MOTORSPORTS',
'FIRD'            : 'FORD',
'ACCORD'          : 'HONDA',          # Honda model
'WORK'            : 'UNKNOWN',
'TAIZHOU ZHILONG' : 'UNKNOWN',
'VIP'             : 'UNKNOWN',
'KAWA'            : 'KAWASAKI',
'TOMAS'           : 'THOMAS',
'HUYUNDAI'        : 'HYUNDAI',
'BENTLY'          : 'BENTLEY',
'CHYR'            : 'CHRYSLER',
'MADZ'            : 'MAZDA',
'MERCED'          : 'MERCEDES-BENZ',
'MITISUBISHI'     : 'MITSUBISHI',
'MITSUBISH'       : 'MITSUBISHI',
'HUDSON'          : 'HUDSON',         # valid historical make
'MOPED'           : 'UNKNOWN',
'MERECEDES'       : 'MERCEDES-BENZ',
'ISZU'            : 'ISUZU',
'CADALIC'         : 'CADILLAC',
'MISSAN'          : 'NISSAN',
'KTM'             : 'KTM',            # motorcycle brand, valid
'MREZ'            : 'MERCEDES-BENZ',
'CHEVROLEY'       : 'CHEVROLET',
'HYUNADAI'        : 'HYUNDAI',
'WOLKSWAGEN'      : 'VOLKSWAGEN',
'SHANGHAI'        : 'UNKNOWN',
'HONAD'           : 'HONDA',
'HARL'            : 'HARLEY-DAVIDSON',
'PORSH'           : 'PORSCHE',
'TAIZHOU ZHONGNE' : 'UNKNOWN',
'MERCEDS'         : 'MERCEDES-BENZ',
'CHEVRLET'        : 'CHEVROLET',
'LEXAS'           : 'LEXUS',
'PLYMOTH'         : 'PLYMOUTH',
'HIUNDAY'         : 'HYUNDAI',
'ROV'             : 'LAND ROVER',
'BMX'             : 'BMW',
'THOMAS BUILT'    : 'THOMAS',
'TELSA'           : 'TESLA',
'WV'              : 'VOLKSWAGEN',
'CHEY'            : 'CHEVROLET',
'WILDFIRE'        : 'UNKNOWN',
'MD'              : 'UNKNOWN',
'TOYPTA'          : 'TOYOTA',
'CADDILLAC'       : 'CADILLAC',
'LEXSUS'          : 'LEXUS',
'TOYOTA SCION'    : 'SCION',
'PORSCH'          : 'PORSCHE',
'APOLLO'          : 'UNKNOWN',
'LINCON'          : 'LINCOLN',
'LEIKE'           : 'UNKNOWN',

# --- ROUND 10 ADDITIONS (10-19 group) ---
'BUIK'       : 'BUICK',
'STLG'       : 'STERLING',
'BNW'        : 'BMW',
'OLDMOBILE'  : 'OLDSMOBILE',
'TPYOTA'     : 'TOYOTA',
'LEXUX'      : 'LEXUS',
'GILG'       : 'GILLIG',
'MITSUBIHI'  : 'MITSUBISHI',
'HIUNDAI'    : 'HYUNDAI',
'HOMD'       : 'HONDA',
'TOYOTSA'    : 'TOYOTA',
'SABB'       : 'SAAB',
'RANGE ROV'  : 'LAND ROVER',
'FRT'        : 'FREIGHTLINER',
'HYUNDAU'    : 'HYUNDAI',
'MITSUB'     : 'MITSUBISHI',
'VOLKSWAG'   : 'VOLKSWAGEN',
'CHEVYROLET' : 'CHEVROLET',
'HYUANDAI'   : 'HYUNDAI',
'INFNITY'    : 'INFINITI',

# --- before bulk marking ---
'DODGE RAM'    : 'DODGE',
'RIVIAN'       : 'RIVIAN',        # valid EV brand
'WESTERN STAR' : 'WESTERN STAR',  # valid truck brand
'LAMBO'        : 'LAMBORGHINI',
'LAMO'         : 'LAMBORGHINI',
'LOTUS'        : 'LOTUS',         # valid make
'ASTON MARTIN' : 'ASTON MARTIN',  # valid make
'KUBOTA'       : 'KUBOTA',        # valid equipment brand
'PIAGGIO'      : 'PIAGGIO',       # valid scooter brand
'WORKHORSE'    : 'WORKHORSE',     # valid truck brand
'VICTORY'      : 'VICTORY',       # motorcycle brand
'CAMRY'        : 'TOYOTA',        # Toyota model
'RAV4'         : 'TOYOTA',        # Toyota model
'ALTIMA'       : 'NISSAN',        # Nissan model
'CIVIC'        : 'HONDA',         # Honda model
'JETTA'        : 'VOLKSWAGEN',    # VW model
'COROLLA'      : 'TOYOTA',        # Toyota model
'ELANTRA'      : 'HYUNDAI',       # Hyundai model
'PASSAT'       : 'VOLKSWAGEN',    # VW model

}

- Step 2 — Define known clean makes 

In [139]:
known_makes = set(make_map.values()) | {
    'KIA', 'GMC', 'AUDI', 'FORD', 'BMW', 'HONDA', 'NISSAN', 'TOYOTA',
    'HYUNDAI', 'CHEVROLET', 'MERCEDES-BENZ', 'SUBARU', 'VOLVO', 'TESLA',
    'LEXUS', 'INFINITI', 'ACURA', 'MAZDA', 'MITSUBISHI', 'VOLKSWAGEN',
    'DODGE', 'JEEP', 'CHRYSLER', 'RAM', 'BUICK', 'CADILLAC', 'LINCOLN',
    'PORSCHE', 'JAGUAR', 'LAND ROVER', 'MINI', 'SATURN', 'PONTIAC',
    'UNKNOWN','SCION', 'MERCURY', 'ISUZU', 'SAAB', 'YAMAHA', 'OLDSMOBILE',
    'INTERNATIONAL', 'KAWASAKI', 'MACK', 'PLYMOUTH', 'FIAT', 'GEO',
    'HUMMER', 'TAOTAO', 'HINO', 'KENWORTH','MASERATI', 'PETERBILT', 
    'HARLEY-DAVIDSON', 'SMART', 'THOMAS', 'TAIZHOU','STERLING', 'UD',
    'DUCATI', 'VESPA', 'KYMCO', 'NEW FLYER', 'GILLIG', 'ALFA ROMEO',
    'BENTLEY', 'ORION', 'GENESIS', 'TOMOS', 'TRIUMPH', 'DAEWOO',
    'WHITE', 'BUELL', 'FERRARI', 'EAGLE', 'AMC', 'HUDSON', 'KTM',
    'RIVIAN', 'WESTERN STAR', 'LAMBORGHINI', 'LOTUS',
    'ASTON MARTIN', 'KUBOTA', 'PIAGGIO', 'WORKHORSE', 'VICTORY'
}

- Step 3 - Clean

In [140]:
# Step 3 - Clean
traffic_df['make_clean'] = (
    traffic_df['make']
    .str.strip()
    .str.upper()
    .replace(make_map)
)


- Step 4 - Bulk mark anything still unmatched

In [141]:

# Step 4 - Bulk mark anything still unmatched
unmatched = traffic_df[~traffic_df['make_clean'].isin(known_makes)]['make_clean'].value_counts()
print(f"Bulk marking {len(unmatched)} values as UNKNOWN")
for val in unmatched.index.tolist():
    make_map[val] = 'UNKNOWN'
    


Bulk marking 2866 values as UNKNOWN


- Filling Missing Values

In [142]:
traffic_df['make_clean'] = traffic_df['make_clean'].fillna('UNKNOWN')

In [143]:
print(traffic_df['make_clean'].isna().sum())  # should be 0
print(f"Total UNKNOWN: {(traffic_df['make_clean'] == 'UNKNOWN').sum():,}")

0
Total UNKNOWN: 6,909


- Step 3 again - Re-run cleaning with bulk marks applied

In [144]:

# Step 3 again - Re-run cleaning with bulk marks applied
traffic_df['make_clean'] = (
    traffic_df['make']
    .str.strip()
    .str.upper()
    .replace(make_map)
)


- Final check

In [145]:

# Final check
unmatched_final = traffic_df[~traffic_df['make_clean'].isin(known_makes)]['make_clean'].value_counts()
print(f"Remaining unmatched: {len(unmatched_final)}")
print(f"Total UNKNOWN: {(traffic_df['make_clean'] == 'UNKNOWN').sum():,}")
print(f"Total rows: {traffic_df.shape[0]:,}")

Remaining unmatched: 0
Total UNKNOWN: 16,180
Total rows: 1,048,575


#### 5.3.8 model

In [146]:
traffic_df['model'].isna().sum()

np.int64(86)

In [147]:
traffic_df['model'].nunique()

16805

In [148]:
traffic_df['model'].value_counts().head(20)

model
4S          96253
TK          60359
CIVIC       38631
ACCORD      38246
CAMRY       33023
COROLLA     29033
ALTIMA      19860
SUV         17869
4DR         17627
4D          15744
2S          14213
VAN         12513
VN          11417
CRV          9868
RAV4         9249
SENTRA       8846
EXPLORER     8815
F150         8558
SONATA       7887
ELANTRA      7813
Name: count, dtype: int64

In [149]:
model_map = {
    # Body style codes → readable categories
    '4S'  : 'SEDAN',
    'SD'  : 'SEDAN',
    '4D'  : 'SEDAN',
    '4DR' : 'SEDAN',
    '2S'  : 'SEDAN',
    '2D'  : 'SEDAN',
    '2DR' : 'SEDAN',

    'TK'  : 'TRUCK',

    'VAN' : 'VAN',
    'VN'  : 'VAN',

    'SUV' : 'SUV',
    'SU'  : 'SUV',
    'UT'  : 'SUV',

    'SW'  : 'STATION WAGON',
    'CP'  : 'COUPE',
    'MC'  : 'MOTORCYCLE',

    # Genuine unknowns
    'CN'  : 'UNKNOWN',
    '4H'  : 'UNKNOWN',

    # Null-like values
    'UNKNOWN' : 'UNKNOWN',
    'N/A'     : 'UNKNOWN',
    'NA'      : 'UNKNOWN',
    'NONE'    : 'UNKNOWN',
    ''        : 'UNKNOWN',

    'CAMRY LE'   : 'CAMRY',
    'CAMRY SE'   : 'CAMRY',
    'SIENNA LE'  : 'SIENNA',
    'CIVIC LX'   : 'CIVIC',
    'CIVIC EX'   : 'CIVIC',
    'CIVIC DX'   : 'CIVIC',
    'ACCORD LX'  : 'ACCORD',
    'ACCORD EX'  : 'ACCORD',
    'ACCORD DX'  : 'ACCORD',

    'SEQUDIA'    : 'SEQUOIA',
    'CI VIC'     : 'CIVIC',
    '4  RUNNER'  : '4RUNNER',
    'CIVIC 4S'   : 'CIVIC',
    '4SACCORD'   : 'ACCORD',
    '4DR  TL'    : 'TL',
    'RSX 2D'     : 'RSX',
    'SW VENZA'   : 'VENZA',
    'I35 4DR'    : 'I35',
    'TK PU DUALLY': 'UNKNOWN',
    'SUV  SOLL'  : 'UNKNOWN',
    'IS2450'     : 'UNKNOWN',
    'TGN'        : 'UNKNOWN',
    'COMMUTER'   : 'UNKNOWN',
    'TRAL HAWK'  : 'UNKNOWN',

    # Pattern 1 — Strip body codes from model names
'CIVIC 4D'       : 'CIVIC',
'CIVIC 4S'       : 'CIVIC',
'CIVIC 4DR'      : 'CIVIC',
'CIVIC 2S'       : 'CIVIC',
'CIVIC 2D'       : 'CIVIC',
'CIVIC 2DR'      : 'CIVIC',
'CIVIC SI'       : 'CIVIC',
'CIVIC 4DOOR'    : 'CIVIC',
'CIV'            : 'CIVIC',
'CIV 4D'         : 'CIVIC',
'CIV 4S'         : 'CIVIC',
'CIVIV'          : 'CIVIC',
'CIVC'           : 'CIVIC',

'ACCORD 4S'      : 'ACCORD',
'ACCORD 4D'      : 'ACCORD',
'ACCORD 4DR'     : 'ACCORD',
'ACCORD 2S'      : 'ACCORD',
'ACCORD 2D'      : 'ACCORD',
'4DR ACCORD'     : 'ACCORD',
'ACC'            : 'ACCORD',
'ACC 4D'         : 'ACCORD',
'ACC 4S'         : 'ACCORD',
'ACORD'          : 'ACCORD',

'CAMRY 4D'       : 'CAMRY',
'CAMRY 4S'       : 'CAMRY',
'4DR CAMRY'      : 'CAMRY',
'4S CAMRY'       : 'CAMRY',
'CAM'            : 'CAMRY',
'CAM 4S'         : 'CAMRY',
'CAMARY'         : 'CAMRY',
'CAMERY'         : 'CAMRY',

'COROLLA 4D'     : 'COROLLA',
'COROLLA 4S'     : 'COROLLA',
'4DR COROLLA'    : 'COROLLA',
'4S COROLLA'     : 'COROLLA',
'CORROLA'        : 'COROLLA',
'CORROLLA'       : 'COROLLA',
'CAROLLA'        : 'COROLLA',
'COROLA'         : 'COROLLA',
'CORLLA'         : 'COROLLA',
'COR'            : 'COROLLA',
'COR 4D'         : 'COROLLA',

'ALTIMA 4S'      : 'ALTIMA',
'ALTIMA 4D'      : 'ALTIMA',
'4DR ALTIMA'     : 'ALTIMA',
'4S ALTIMA'      : 'ALTIMA',
'ALT'            : 'ALTIMA',

'SENTRA 4D'      : 'SENTRA',
'SENTRA 4S'      : 'SENTRA',
'4S SENTRA'      : 'SENTRA',

'JETTA 4S'       : 'JETTA',
'JETTA 4D'       : 'JETTA',
'4S JETTA'       : 'JETTA',

'ELANTRA 4D'     : 'ELANTRA',
'ELANTRA 4S'     : 'ELANTRA',
'4S ELANTRA'     : 'ELANTRA',
'ELENTRA'        : 'ELANTRA',

'SONATA 4D'      : 'SONATA',
'SONATA 4S'      : 'SONATA',
'4S SONATA'      : 'SONATA',
'SONOTA'         : 'SONATA',
'SONTA'          : 'SONATA',
'SON'            : 'SONATA',

'IMPALA 4S'      : 'IMPALA',
'4S IMPALA'      : 'IMPALA',

'FOCUS 4D'       : 'FOCUS',
'FOCUS 4S'       : 'FOCUS',
'4S FOCUS'       : 'FOCUS',

'MALIBU 4S'      : 'MALIBU',
'4S MALIBU'      : 'MALIBU',

'FUSION 4D'      : 'FUSION',

'TL 4S'          : 'TL',
'4S TL'          : 'TL',
'4DR  TL'        : 'TL',

'AVALON 4S'      : 'AVALON',
'4S AVALON'      : 'AVALON',

'PRIUS 4D'       : 'PRIUS',
'PRUIS'          : 'PRIUS',

'MAXIMA 4S'      : 'MAXIMA',
'MAXIMA 4D'      : 'MAXIMA',
'4S MAXIMA'      : 'MAXIMA',

'PILOT SUV'      : 'PILOT',
'MDX SUV'        : 'MDX',
'CRV SUV'        : 'CRV',
'RAV4 SUV'       : 'RAV4',
'RAV 4'          : 'RAV4',
'RAV-4'          : 'RAV4',
'SUV RAV-4'      : 'RAV4',
'SUV CR-V'       : 'CRV',
'CR-V'           : 'CRV',
'HR-V'           : 'HRV',
'CX-5'           : 'CX5',
'CX-7'           : 'CX7',
'CX-9'           : 'CX9',
'F-150'          : 'F150',
'F-250'          : 'F250',
'F-350'          : 'F350',
'TK F150'        : 'F150',
'F150 TK'        : 'F150',

'EXPLORER SUV'   : 'EXPLORER',
'EXPLORER TK'    : 'EXPLORER',
'SUV EXPLORER'   : 'EXPLORER',
'EXPIDITION'     : 'EXPEDITION',

'TRAIL BLAZER'   : 'TRAILBLAZER',
'GRAND CHEROKEE' : 'GRAND CHEROKEE',
'GRAND CHEREKEE' : 'GRAND CHEROKEE',
'GRAND CHER'     : 'GRAND CHEROKEE',
'GR CHEROKEE'    : 'GRAND CHEROKEE',
'GRAND CHEROK'   : 'GRAND CHEROKEE',

'CROWN VIC'      : 'CROWN VICTORIA',
'SANTE FE'       : 'SANTA FE',
'SANTAFE'        : 'SANTA FE',
'TUSCON'         : 'TUCSON',

'ODESSEY'        : 'ODYSSEY',
'ODESSY'         : 'ODYSSEY',
'ODDYSEY'        : 'ODYSSEY',
'ODYSSEY VAN'    : 'ODYSSEY',
'ODYSSEY VN'     : 'ODYSSEY',
'VN ODYSSEY'     : 'ODYSSEY',
'SIENNA VAN'     : 'SIENNA',
'SIENNA VN'      : 'SIENNA',
'SIENA'          : 'SIENNA',

'TARUS'          : 'TAURUS',
'TOWN & COUNTRY' : 'TOWN & COUNTRY',
'TOWN AND COUNTR': 'TOWN & COUNTRY',
'TOWN COUNTRY'   : 'TOWN & COUNTRY',
'TOWN/COUNTRY'   : 'TOWN & COUNTRY',
'TOWN&COUNTRY'   : 'TOWN & COUNTRY',
'TOWNCAR'        : 'TOWN CAR',

'FORESTER'       : 'FORESTER',
'FORRESTER'      : 'FORESTER',
'OUTBACK SW'     : 'OUTBACK',

'4 RUNNER'       : '4RUNNER',
'4 DR'           : 'SEDAN',
'4DOOR'          : 'SEDAN',
'4 DOOR'         : 'SEDAN',
'FOUR DOOR'      : 'SEDAN',
'2DOOR'          : 'SEDAN',
'2 DOOR'         : 'SEDAN',
'5D'             : 'SEDAN',
'5S'             : 'SEDAN',

'PICKUP'         : 'TRUCK',
'PICK UP'        : 'TRUCK',
'PICK-UP'        : 'TRUCK',
'PU'             : 'TRUCK',
'PK'             : 'TRUCK',
'TRK'            : 'TRUCK',
'PKTK'           : 'TRUCK',
'PU TK'          : 'TRUCK',
'RAM TK'         : 'TRUCK',
'PICKUP TRUCK'   : 'TRUCK',
'DUMP TRUCK'     : 'TRUCK',
'DUMP TK'        : 'TRUCK',
'DUMP'           : 'TRUCK',
'TOW TRUCK'      : 'TRUCK',
'TOW TK'         : 'TRUCK',
'TRASH TRUCK'    : 'TRUCK',
'BOX TRUCK'      : 'TRUCK',
'BOX TK'         : 'TRUCK',
'BOX'            : 'TRUCK',
'BX TK'          : 'TRUCK',

'MINIVAN'        : 'VAN',
'MINI VAN'       : 'VAN',
'EXPRESS VAN'    : 'VAN',
'EXPRESS VN'     : 'VAN',
'ASTRO VAN'      : 'VAN',
'ASTROVAN'       : 'VAN',
'E250 VAN'       : 'VAN',
'E150 VAN'       : 'VAN',
'E350 VAN'       : 'VAN',
'RAM VAN'        : 'VAN',
'WORK VAN'       : 'VAN',
'CARGO VAN'      : 'VAN',
'TRANSIT VAN'    : 'VAN',
'M VAN'          : 'VAN',
'W VAN'          : 'VAN',

'GALLANT'        : 'GALANT',
'CAMERO'         : 'CAMARO',
'SEABRING'       : 'SEBRING',
'SORRENTO'       : 'SORENTO',
'PROTOGE'        : 'PROTEGE',
'TAKOMA'         : 'TACOMA',
'TOCOMA'         : 'TACOMA',
'TACOMA PU'      : 'TACOMA',
'TACOMA TK'      : 'TACOMA',
'ROUGE'          : 'ROGUE',
'EXTERRA'        : 'XTERRA',
'XTERA'          : 'XTERRA',
'SEQUIOA'        : 'SEQUOIA',
'COLBALT'        : 'COBALT',
'ARCADIA'        : 'ACADIA',
'CAROLLA'        : 'COROLLA',
'VELOSTAR'       : 'VELOSTER',
'CIVIV'          : 'CIVIC',
'WINSTAR'        : 'WINDSTAR',
'LASABRE'        : 'LESABRE',
'LE SABRE'       : 'LESABRE',
'MONTECARLO'     : 'MONTE CARLO',
'LANDCRUISER'    : 'LAND CRUISER',
'SCION TC'       : 'TC',
'SCION XB'       : 'XB',
'SCION FRS'      : 'FRS',
'MAZDA3'         : 'MAZDA 3',
'MAZDA6'         : 'MAZDA 6',
'RAM1500'        : 'RAM 1500',
'RAM 2500'       : 'RAM 2500',
'RAM 3500'       : 'RAM 3500',
'ML 350'         : 'ML350',
'ML 320'         : 'ML320',
'ES 350'         : 'ES350',
'ES 300'         : 'ES300',
'ES 330'         : 'ES330',
'GS 350'         : 'GS350',
'GS 300'         : 'GS300',
'IS 250'         : 'IS250',
'IS 300'         : 'IS300',
'CLA 250'        : 'CLA250',
'GLA 250'        : 'GLA250',
'GLK 350'        : 'GLK350',
'GLE 350'        : 'GLE350',
'C 300'          : 'C300',
'3.2TL'          : '3.2 TL',
'3.2 TL'         : 'TL',
'3.5RL'          : 'RL',

# Still unknown
'UNK'            : 'UNKNOWN',
'SDN'            : 'UNKNOWN',
'HB'             : 'UNKNOWN',
'HY'             : 'UNKNOWN',
'DS'             : 'UNKNOWN',
'DT'             : 'UNKNOWN',
'RD'             : 'UNKNOWN',
'TR'             : 'UNKNOWN',
'PC'             : 'UNKNOWN',
'WG'             : 'UNKNOWN',
'CG'             : 'UNKNOWN',
'BU'             : 'UNKNOWN',
'MV'             : 'UNKNOWN',
'CV'             : 'UNKNOWN',
'PV'             : 'UNKNOWN',
'CC'             : 'UNKNOWN',
'CL'             : 'UNKNOWN',
'GC'             : 'UNKNOWN',
'RS'             : 'UNKNOWN',
'SE'             : 'UNKNOWN',
'ST'             : 'UNKNOWN',
'SI'             : 'UNKNOWN',
'LS'             : 'UNKNOWN',
'GT'             : 'UNKNOWN',
'DX'             : 'UNKNOWN',
'LX'             : 'UNKNOWN',
'XX'             : 'UNKNOWN',
'X'              : 'UNKNOWN',
'Y'              : 'UNKNOWN',
'2'              : 'UNKNOWN',
'5'              : 'UNKNOWN',
'S'              : 'UNKNOWN',
}

- cleaning and handling missing values

In [150]:
traffic_df['model_clean'] = (
    traffic_df['model']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
    .replace(model_map)
)

-validation

In [151]:
# Final model validation
print(f"Total rows:        {traffic_df.shape[0]:,}")
print(f"UNKNOWN:           {(traffic_df['model_clean'] == 'UNKNOWN').sum():,} ({(traffic_df['model_clean'] == 'UNKNOWN').mean()*100:.2f}%)")
print(f"NaN remaining:     {traffic_df['model_clean'].isna().sum()}")
print(f"Unique models:     {traffic_df['model_clean'].nunique():,}")
print(f"\nBody style counts:")
for val in ['SEDAN', 'TRUCK', 'SUV', 'VAN', 'COUPE', 'STATION WAGON', 'MOTORCYCLE']:
    count = (traffic_df['model_clean'] == val).sum()
    print(f"  {val:15} {count:,}")

Total rows:        1,048,575
UNKNOWN:           17,357 (1.66%)
NaN remaining:     0
Unique models:     16,519

Body style counts:
  SEDAN           165,462
  TRUCK           75,364
  SUV             27,943
  VAN             26,058
  COUPE           1,941
  STATION WAGON   6,455
  MOTORCYCLE      2,326


#### 5.3.9 color

In [152]:
print(f"NaN count: {traffic_df['color'].isna().sum():,}")
print(f"Unique colors: {traffic_df['color'].nunique():,}")
print(f"\nTop 30 colors:")
print(traffic_df['color'].value_counts().head(30))
print(f"\nBottom 20 colors:")
print(traffic_df['color'].value_counts().tail(20))


NaN count: 13,772
Unique colors: 26

Top 30 colors:
color
BLACK          220416
SILVER         179860
WHITE          173221
GRAY           125555
RED             79411
BLUE            76858
GREEN           33524
GOLD            30050
BLUE, DARK      22384
MAROON          18935
TAN             18468
BLUE, LIGHT     12068
GREEN, DK       10176
BEIGE            9786
GREEN, LGT       5315
BROWN            5059
YELLOW           3807
ORANGE           3644
BRONZE           2265
PURPLE           2000
MULTICOLOR        825
CREAM             688
COPPER            297
PINK              160
CAMOUFLAGE         17
CHROME             14
Name: count, dtype: int64

Bottom 20 colors:
color
GREEN          33524
GOLD           30050
BLUE, DARK     22384
MAROON         18935
TAN            18468
BLUE, LIGHT    12068
GREEN, DK      10176
BEIGE           9786
GREEN, LGT      5315
BROWN           5059
YELLOW          3807
ORANGE          3644
BRONZE          2265
PURPLE          2000
MULTICOLOR       825
CREA

In [153]:
color_map = {
    # Standardize naming convention
    'BLUE, DARK'  : 'DARK BLUE',
    'BLUE, LIGHT' : 'LIGHT BLUE',
    'GREEN, DK'   : 'DARK GREEN',
    'GREEN, LGT'  : 'LIGHT GREEN',
}


In [154]:

# Apply cleaning
traffic_df['color_clean'] = (
    traffic_df['color']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
    .replace(color_map)
)


In [155]:

# Validation
print(f"Total rows:     {traffic_df.shape[0]:,}")
print(f"Unique colors:  {traffic_df['color_clean'].nunique():,}")
print(f"UNKNOWN:        {(traffic_df['color_clean'] == 'UNKNOWN').sum():,} ({(traffic_df['color_clean'] == 'UNKNOWN').mean()*100:.2f}%)")
print(f"NaN remaining:  {traffic_df['color_clean'].isna().sum()}")
print(f"\nAll colors:")
print(traffic_df['color_clean'].value_counts())

Total rows:     1,048,575
Unique colors:  27
UNKNOWN:        13,772 (1.31%)
NaN remaining:  0

All colors:
color_clean
BLACK          220416
SILVER         179860
WHITE          173221
GRAY           125555
RED             79411
BLUE            76858
GREEN           33524
GOLD            30050
DARK BLUE       22384
MAROON          18935
TAN             18468
UNKNOWN         13772
LIGHT BLUE      12068
DARK GREEN      10176
BEIGE            9786
LIGHT GREEN      5315
BROWN            5059
YELLOW           3807
ORANGE           3644
BRONZE           2265
PURPLE           2000
MULTICOLOR        825
CREAM             688
COPPER            297
PINK              160
CAMOUFLAGE         17
CHROME             14
Name: count, dtype: int64


#### 5.3.10 driver_city

In [156]:
print(f"NaN count: {traffic_df['driver_city'].isna().sum():,}")
print(f"Unique city: {traffic_df['driver_city'].nunique():,}")
print(f"\nTop 30 cities:")
print(traffic_df['driver_city'].value_counts().head(30))
print(f"\nBottom 20 cities:")
print(traffic_df['driver_city'].value_counts().tail(20))

NaN count: 170
Unique city: 6,373

Top 30 cities:
driver_city
SILVER SPRING         250215
GAITHERSBURG          110116
GERMANTOWN             94709
ROCKVILLE              80782
WASHINGTON             33145
BETHESDA               27833
MONTGOMERY VILLAGE     27490
HYATTSVILLE            26917
POTOMAC                19070
FREDERICK              16223
CLARKSBURG             16193
OLNEY                  16179
LAUREL                 15172
DAMASCUS               12557
NORTH POTOMAC          10348
TAKOMA PARK            10299
KENSINGTON             10213
BURTONSVILLE            9596
BELTSVILLE              9463
BALTIMORE               8547
CHEVY CHASE             8424
BOYDS                   7154
DERWOOD                 7013
COLUMBIA                6923
BOWIE                   6093
POOLESVILLE             5918
BROOKEVILLE             5609
WHEATON                 5547
ADELPHI                 5118
MOUNT AIRY              4891
Name: count, dtype: int64

Bottom 20 cities:
driver_city
MAULDIN    

In [157]:
city_counts = traffic_df['driver_city'].str.strip().str.upper().value_counts()

print(f"Total unique cities: {len(city_counts):,}")
print(f"Cities 100+ occurrences: {(city_counts >= 100).sum():,}")
print(f"Cities 10-99 occurrences: {((city_counts >= 10) & (city_counts < 100)).sum():,}")
print(f"Cities < 10 occurrences:  {(city_counts < 10).sum():,}")

Total unique cities: 6,373
Cities 100+ occurrences: 273
Cities 10-99 occurrences: 983
Cities < 10 occurrences:  5,117


- 100+ occurrences:    273 cities  → worth fixing manually
- 10-99 occurrences:   983 cities  → fix obvious ones
- < 10 occurrences:  5,117 cities  → bulk mark UNKNOWN

In [158]:
# Normalize first
traffic_df['driver_city_clean'] = (
    traffic_df['driver_city']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
)

In [159]:
# Show 100+ group
city_counts = traffic_df['driver_city_clean'].value_counts()
print(city_counts[city_counts >= 100].to_string())

driver_city_clean
SILVER SPRING         250215
GAITHERSBURG          110116
GERMANTOWN             94709
ROCKVILLE              80782
WASHINGTON             33145
BETHESDA               27833
MONTGOMERY VILLAGE     27490
HYATTSVILLE            26917
POTOMAC                19070
FREDERICK              16223
CLARKSBURG             16193
OLNEY                  16179
LAUREL                 15172
DAMASCUS               12557
NORTH POTOMAC          10348
TAKOMA PARK            10299
KENSINGTON             10213
BURTONSVILLE            9596
BELTSVILLE              9463
BALTIMORE               8547
CHEVY CHASE             8424
BOYDS                   7154
DERWOOD                 7013
COLUMBIA                6923
BOWIE                   6093
POOLESVILLE             5918
BROOKEVILLE             5609
WHEATON                 5547
ADELPHI                 5118
MOUNT AIRY              4891
UPPER MARLBORO          4854
LANHAM                  4824
ALEXANDRIA              4364
GREENBELT               3

In [160]:
city_map = {
    'MONTGOMRY VLG'  : 'MONTGOMERY VILLAGE',
    'MONT VILLAGE'   : 'MONTGOMERY VILLAGE',
    'N POTOMAC'      : 'NORTH POTOMAC',
    'N. POTOMAC'     : 'NORTH POTOMAC',
    'N BETHESDA'     : 'NORTH BETHESDA',
    'N. BETHESDA'    : 'NORTH BETHESDA',
    'GAITHERBURG'    : 'GAITHERSBURG',
    'HYATSVILLE'     : 'HYATTSVILLE',
    'SILVERSPRING'   : 'SILVER SPRING',
    'CAPITAL HEIGHTS': 'CAPITOL HEIGHTS',
    'MT AIRY'        : 'MOUNT AIRY',
    'MT. AIRY'       : 'MOUNT AIRY',
    'FT WASHINGTON'  : 'FORT WASHINGTON',
    'MOUNT RAINER'   : 'MOUNT RAINIER',
    'TACOMA PARK'    : 'TAKOMA PARK',
    'NEW CARROLTON'  : 'NEW CARROLLTON',
    'ELLICOT CITY'   : 'ELLICOTT CITY',
    'WASHINGTON DC'  : 'WASHINGTON',
    'NW WASHINGTON'  : 'WASHINGTON',
    'XX'             : 'UNKNOWN',
    'NW'             : 'UNKNOWN',
    'LAURAL'               : 'LAUREL',
    'SILVER SRING'         : 'SILVER SPRING',
    'MONT. VILLAGE'        : 'MONTGOMERY VILLAGE',
    'GERMATOWN'            : 'GERMANTOWN',
    'DC'                   : 'WASHINGTON',
    'ROCKVILE'             : 'ROCKVILLE',
    'NE'                   : 'UNKNOWN',
    'NORTH WEST'           : 'UNKNOWN',
    'OWINGS MILL'          : 'OWINGS MILLS',
    'CENTERVILLE'          : 'CENTREVILLE',
    'LUTHERVILLE TIMONIUM' : 'LUTHERVILLE',
}


In [161]:

# Apply
traffic_df['driver_city_clean'] = (
    traffic_df['driver_city']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
    .replace(city_map)
)


In [162]:

# Check remaining issues in 10-99 group
city_counts = traffic_df['driver_city_clean'].value_counts()
print(f"Unique cities: {len(city_counts):,}")
print(f"\n10-99 group sample:")
print(city_counts[(city_counts >= 10) & (city_counts < 100)].head(50).to_string())

Unique cities: 6,341

10-99 group sample:
driver_city_clean
HAMILTON             99
CHESAPEAKE BEACH     99
KEEDYSVILLE          98
FAIRMOUNT HEIGHTS    98
EASTON               98
COOKSVILLE           97
CHARLOTTESVILLE      96
HAMPTON              96
STEVENSVILLE         95
JOPPA                95
SHARPSBURG           94
PRINCE FREDERICK     94
ABERDEEN             94
ONLEY                93
DUNKIRK              90
KEYMAR               88
HENRICO              88
SAN ANTONIO          86
HOLLYWOOD            84
CAMP SPRINGS         82
TIMONIUM             82
BOCA RATON           81
LANCASTER            80
MORNINGSIDE          78
GREENVILLE           77
ROCHESTER            77
GLENDALE             75
AUSTIN               75
CAMBRIDGE            73
GREENCASTLE          73
INWOOD               73
GRASONVILLE          72
LANGLEY PARK         72
BERLIN               71
SEATTLE              71
KEARNEYSVILLE        69
PHOENIX              68
TUCSON               67
KING GEORGE          66
LANS

In [163]:
# Show cities in 10-99 group that are NOT in known clean list
known_clean_cities = set(city_map.values()) | {
    'SILVER SPRING', 'GAITHERSBURG', 'GERMANTOWN', 'ROCKVILLE',
    'WASHINGTON', 'BETHESDA', 'MONTGOMERY VILLAGE', 'HYATTSVILLE',
    'POTOMAC', 'FREDERICK', 'CLARKSBURG', 'OLNEY', 'LAUREL',
    'DAMASCUS', 'NORTH POTOMAC', 'TAKOMA PARK', 'KENSINGTON',
    'BURTONSVILLE', 'BELTSVILLE', 'BALTIMORE', 'CHEVY CHASE',
    # add more as you go
}

city_counts = traffic_df['driver_city_clean'].value_counts()
unclean = city_counts[
    (city_counts >= 10) & 
    (city_counts < 100) & 
    (~city_counts.index.isin(known_clean_cities))
]
print(f"Remaining unclean cities (10-99): {len(unclean)}")
print(unclean.head(50).to_string())

Remaining unclean cities (10-99): 971
driver_city_clean
HAMILTON             99
CHESAPEAKE BEACH     99
KEEDYSVILLE          98
FAIRMOUNT HEIGHTS    98
EASTON               98
COOKSVILLE           97
CHARLOTTESVILLE      96
HAMPTON              96
STEVENSVILLE         95
JOPPA                95
SHARPSBURG           94
PRINCE FREDERICK     94
ABERDEEN             94
ONLEY                93
DUNKIRK              90
KEYMAR               88
HENRICO              88
SAN ANTONIO          86
HOLLYWOOD            84
CAMP SPRINGS         82
TIMONIUM             82
BOCA RATON           81
LANCASTER            80
MORNINGSIDE          78
GREENVILLE           77
ROCHESTER            77
GLENDALE             75
AUSTIN               75
CAMBRIDGE            73
GREENCASTLE          73
INWOOD               73
GRASONVILLE          72
LANGLEY PARK         72
BERLIN               71
SEATTLE              71
KEARNEYSVILLE        69
PHOENIX              68
TUCSON               67
KING GEORGE          66
LANSDOWN

In [164]:
city_counts = traffic_df['driver_city_clean'].value_counts()

print(f"Cities 50+:   {(city_counts >= 50).sum():,}  cities")
print(f"Cities 10-49: {((city_counts >= 10) & (city_counts < 50)).sum():,}  cities")
print(f"Cities < 10:  {(city_counts < 10).sum():,}  cities")

print(f"\nRows affected if we bulk mark < 10:  {city_counts[city_counts < 10].sum():,}")
print(f"Rows affected if we bulk mark < 20:  {city_counts[city_counts < 20].sum():,}")
print(f"Rows affected if we bulk mark < 50:  {city_counts[city_counts < 50].sum():,}")

Cities 50+:   357  cities
Cities 10-49: 867  cities
Cities < 10:  5,117  cities

Rows affected if we bulk mark < 10:  13,584
Rows affected if we bulk mark < 20:  20,885
Rows affected if we bulk mark < 50:  30,636


In [165]:
# Bulk mark cities with < 10 occurrences
city_counts = traffic_df['driver_city_clean'].value_counts()
rare_cities = city_counts[city_counts < 10].index.tolist()

for val in rare_cities:
    city_map[val] = 'UNKNOWN'

# Re-run cleaning
traffic_df['driver_city_clean'] = (
    traffic_df['driver_city']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
    .replace(city_map)
)

# Final validation
print(f"Total rows:      {traffic_df.shape[0]:,}")
print(f"Unique cities:   {traffic_df['driver_city_clean'].nunique():,}")
print(f"UNKNOWN:         {(traffic_df['driver_city_clean'] == 'UNKNOWN').sum():,} ({(traffic_df['driver_city_clean'] == 'UNKNOWN').mean()*100:.2f}%)")
print(f"NaN remaining:   {traffic_df['driver_city_clean'].isna().sum()}")
print(f"\nTop 20 cities:")
print(traffic_df['driver_city_clean'].value_counts().head(20))

Total rows:      1,048,575
Unique cities:   1,224
UNKNOWN:         14,615 (1.39%)
NaN remaining:   0

Top 20 cities:
driver_city_clean
SILVER SPRING         250451
GAITHERSBURG          110447
GERMANTOWN             94788
ROCKVILLE              80852
WASHINGTON             33702
MONTGOMERY VILLAGE     28284
BETHESDA               27833
HYATTSVILLE            27207
POTOMAC                19070
FREDERICK              16223
CLARKSBURG             16193
OLNEY                  16179
LAUREL                 15264
UNKNOWN                14615
DAMASCUS               12557
NORTH POTOMAC          11149
TAKOMA PARK            10431
KENSINGTON             10213
BURTONSVILLE            9596
BELTSVILLE              9463
Name: count, dtype: int64


1. Normalized    — strip + upper
2. Fixed NaNs    — fillna UNKNOWN
3. Fixed typos   — 20+ corrections
                   SILVERSPRING → SILVER SPRING
                   HYATSVILLE → HYATTSVILLE
                   GAITHERBURG → GAITHERSBURG etc.
4. Standardized  — DC/WASHINGTON DC → WASHINGTON
                   MT AIRY/MT. AIRY → MOUNT AIRY
                   N POTOMAC/N. POTOMAC → NORTH POTOMAC etc.
5. Bulk marked   — 5,117 cities < 10 occurrences → UNKNOWN

1. Normalized    — strip + upper
2. Fixed NaNs    — fillna UNKNOWN
3. Fixed typos   — 20+ corrections
                   SILVERSPRING → SILVER SPRING
                   HYATSVILLE → HYATTSVILLE
                   GAITHERBURG → GAITHERSBURG etc.
4. Standardized  — DC/WASHINGTON DC → WASHINGTON
                   MT AIRY/MT. AIRY → MOUNT AIRY
                   N POTOMAC/N. POTOMAC → NORTH POTOMAC etc.
5. Bulk marked   — 5,117 cities < 10 occurrences → UNKNOWN

#### 5.3.11 driver_vehicle_df table creation

- Create a table with:
    + One row per driver/vehicle record
    + Linked to traffic_stop via stop_id
    + Has its own primary key → driver_id

##### Step 1: Create dataframe using _clean columns

In [166]:
driver_vehicle_cols = [
    'stop_id',
    'race', 'gender',
    'driver_city_clean', 'driver_state', 'dl_state',
    'commercial_license', 'hazmat', 'commercial_vehicle',
    'alcohol', 'work_zone',
    'vehicle_code', 'vehicle_category',
    'year', 'make_clean', 'model_clean', 'color_clean'
]

driver_vehicle_df = traffic_df[driver_vehicle_cols].copy()

##### Step 2: Rename inside the new dataframe

In [167]:
driver_vehicle_df.rename(columns={
    'driver_city_clean': 'driver_city',
    'make_clean': 'make',
    'model_clean': 'model',
    'color_clean': 'color'
}, inplace=True)

##### Step 3: Drop duplicates

In [168]:
driver_vehicle_df.duplicated().sum()

np.int64(480258)

In [169]:
driver_vehicle_df.shape

(1048575, 17)

In [170]:
driver_vehicle_df = driver_vehicle_df.drop_duplicates().reset_index(drop=True)

In [171]:
driver_vehicle_df.insert(0, 'driver_id', range(1, len(driver_vehicle_df)+1))

In [172]:
driver_vehicle_df.head(5)

,driver_id,stop_id,race,gender,driver_city,driver_state,dl_state,commercial_license,hazmat,commercial_vehicle,alcohol,work_zone,vehicle_code,vehicle_category,year,make,model,color
0,1,1,WHITE,M,GAITHERSBURG,MD,MD,0,0,0,0,0,02,AUTOMOBILE,2007,CHEVROLET,CRUZ,BLACK
1,2,2,HISPANIC,M,GAITHERSBURG,MD,MD,0,0,0,0,0,02,AUTOMOBILE,2005,FORD,EXPLORER,BLACK
2,3,3,WHITE,F,SILVER SPRING,MD,MD,0,0,0,0,0,02,AUTOMOBILE,2013,HYUNDAI,SONATA,RED
3,4,4,WHITE,M,BETHESDA,MD,MD,0,0,0,0,0,19,MOPED,2020,UNKNOWN,SXR,RED
4,5,5,BLACK,M,GAITHERSBURG,MD,MD,0,0,0,0,0,02,AUTOMOBILE,2019,HONDA,SEDAN,GRAY


In [173]:
# ============================================================
# DRIVER VEHICLE — DDL PREPARATION
# ============================================================

print(f"Total rows:        {driver_vehicle_df.shape[0]:,}")
print(f"Unique driver_ids: {driver_vehicle_df['driver_id'].nunique():,}")
print(f"NaN driver_ids:    {driver_vehicle_df['driver_id'].isna().sum():,}")
print(f"Is driver_id unique: {driver_vehicle_df['driver_id'].nunique() == driver_vehicle_df.shape[0]}")

# Check varchar lengths
str_cols = ['race', 'gender', 'driver_city', 'driver_state', 
            'dl_state', 'vehicle_code', 'vehicle_category', 
            'make', 'model', 'color']
print(f"\nMax lengths:")
for col in str_cols:
    max_len = driver_vehicle_df[col].str.len().max()
    print(f"  {col:20} max length: {max_len}")

# Check dtypes
print(f"\nDtypes:")
print(driver_vehicle_df.dtypes)

# Check NaN counts
print(f"\nNaN counts:")
print(driver_vehicle_df.isna().sum())

Total rows:        568,317
Unique driver_ids: 568,317
NaN driver_ids:    0
Is driver_id unique: True

Max lengths:
  race                 max length: 15
  gender               max length: 1
  driver_city          max length: 20
  driver_state         max length: 7
  dl_state             max length: 7
  vehicle_code         max length: 2
  vehicle_category     max length: 22
  make                 max length: 15.0
  model                max length: 21
  color                max length: 11

Dtypes:
driver_id              int64
stop_id                int64
race                  object
gender                object
driver_city           object
driver_state          object
dl_state              object
commercial_license     int64
hazmat                 int64
commercial_vehicle     int64
alcohol                int64
work_zone              int64
vehicle_code          object
vehicle_category      object
year                   Int64
make                  object
model                 object
color

In [174]:
# Check why make max length is float
print(driver_vehicle_df['make'].str.len().describe())
print(f"\nSample makes:")
print(driver_vehicle_df['make'].value_counts().head(10))

count    568298.000000
mean          6.021668
std           2.104901
min           2.000000
25%           5.000000
50%           6.000000
75%           7.000000
max          15.000000
Name: make, dtype: float64

Sample makes:
make
TOYOTA           101559
HONDA             83965
FORD              52075
NISSAN            40485
CHEVROLET         40246
HYUNDAI           20328
ACURA             17264
DODGE             17228
BMW               16835
MERCEDES-BENZ     16832
Name: count, dtype: int64


## 5.4 Violation_Charge 

- General cleaning (apply to all text columns)

In [175]:
violation_cols = [
    'description', 'violation_type',
    'charge', 'article', 'search_arrest_reason'
]

for col in violation_cols:
    traffic_df[col] = (
        traffic_df[col]
        .astype(str)
        .str.upper()
        .str.strip()
    )

#### 5.4.1 description

In [176]:
print(f"NaN count: {traffic_df['description'].isna().sum():,}")
print(f"Unique : {traffic_df['description'].nunique():,}")
print(f"\nTop 30 descriptions:")
print(traffic_df['description'].value_counts().head(30))
print(f"\nBottom 20 descriptions:")
print(traffic_df['description'].value_counts().tail(20))

NaN count: 0
Unique : 11,195

Top 30 descriptions:
description
DRIVER FAILURE TO OBEY PROPERLY PLACED TRAFFIC CONTROL DEVICE INSTRUCTIONS                           70435
FAILURE TO DISPLAY REGISTRATION CARD UPON DEMAND BY POLICE OFFICER                                   38406
FAILURE OF INDIVIDUAL DRIVING ON HIGHWAY TO DISPLAY LICENSE TO UNIFORMED POLICE ON DEMAND            36029
PERSON DRIVING MOTOR VEHICLE ON HIGHWAY OR PUBLIC USE PROPERTY ON SUSPENDED LICENSE AND PRIVILEGE    34475
DRIVING VEHICLE ON HIGHWAY WITH SUSPENDED REGISTRATION                                               30001
DRIVER USING HANDS TO USE HANDHELD TELEPHONE WHILEMOTOR VEHICLE IS IN MOTION                         24936
DRIVING VEHICLE WHILE UNDER THE INFLUENCE OF ALCOHOL                                                 24286
DRIVING MOTOR VEHICLE ON HIGHWAY WITHOUT REQUIRED LICENSE AND AUTHORIZATION                          24188
DISPLAYING EXPIRED REGISTRATION PLATE ISSUED BY ANY STATE                        

- Step1 : Categories 

In [177]:
categories = {
    'SPEED'          : r'SPEED|MPH|SPEEDING',
    'DUI'            : r'ALCOHOL|IMPAIR|INFLUENCE|DUI|DWI',
    'LICENSE'        : r'LICENSE|SUSPEND|LEARNER|PERMIT|LIC\.|SUSP\.|PROVISIONAL|REVOKED|RESTRICTION|FICTITIOUS|SUSP$',
    'REGISTRATION'   : r'REGISTRAT|PLATES|TABS|UNREGISTERED|REG\. PLATE|PLATE ILLUMINATION|DISPLAYING REG|CURRENT TAGS|DISPLAY.*REG|REG\. CARD|VEHICLE I\.D|RENTAL|UNREG\.|DECAL|SURENDER REG|APPLY.*REG|TITLE CERT|PLATE.*COVER',
    'SEATBELT'       : r'SEATBELT|SEAT BELT|RESTRAIN|CHILD SAFETY|CHILD SEAT',
    'PHONE'          : r'PHONE|HANDHELD|TELEPHONE|DEVICE|TEXT|ELECTRONIC MSG',
    'RECKLESS'       : r'RECKLESS|CARELESS|NEGLIGENT|WANTON|AGGRESSIVE|SPINNING WHEELS|ELUDE|EXCESSIVE NOISE|EARPLUGS',
    'STOP SIGN'      : r'STOP SIGN|STOP AT',
    'RED LIGHT'      : r'RED SIGNAL|RED LIGHT|TRAFFIC SIGNAL|CIRCULAR RED',
    'LANE'           : r'LANE|CHANGING LANES|RIGHT HALF|WRONG WAY|ONE WAY|RIGHT OF CENTER|CURB|SIDEWALK|DIVIDED HWY|RIGHT HAND|ENTERING HWY|PRIVATE DRIVEWAY',
    'INSURANCE'      : r'INSUR|UNINSURED|SECURITY',
    'EQUIPMENT'      : r'EQUIPMENT|HEADLIGHT|TAILLIGHT|BRAKE|LAMP|TINT|WINDOW|STOP LIGHTS|TAG LIGHTS|EXHAUST|HORN|WINDSHIELD|UNSAFE VEH|GLASS|TIRES|TIRE|GLARING LIGHT|LIGHT DISTRIBUTION|STOPLIGHT|EARPLUGS|NOISE|INSPECTION|BRAKING|MIRROR|VISION|TV.*EQUIP|VIDEO.*EQUIP|FLASHING|SOUND|AMPLIF|LOADED VEH|UNSECURED',
    'YIELD'          : r'YIELD|RIGHT OF WAY|PEDESTRIAN|CROSSWALK',
    'FOLLOWING'      : r'FOLLOWING|TAILGAT',
    'PARKING'        : r'PARKING|PARK|OBSTRUCT|STOPPING VEH|INTERSECTION|PROHIBITED.*SIGN|ABANDON|ENTERING HIGHWAY|APPROACH BY POLICE',
    'HIT AND RUN'    : r'SCENE OF ACCIDENT|HIT AND RUN|REMAIN AT|AFTER ACCIDENT|UNATTENDED VEH|FURNISH REQ ID|UNATTENDED PROPERTY|RENDER.*ASSIST|FURNISH.*ID|NOTIFY OWNER|RPT.*DAMAGE|EXHIBIT LIC|ATTEND VEH|NEAREST POLICE',
    'IMPROPER TURN'  : r'TURN|TURNING|BACKING|PASSING',
    'FRAUD'          : r'FALSIF|FRAUDULENT|FALSE.*NAME|FICTITIOUS|FALSE ACCIDENT|DISOBEYING|POLICE OFFICER|HISTORIC|WITHOUT.*CONSENT|DEPRIVE OWNER|TAKING VEH|RACE ON HWY|ENGLISH SPEAKING|REFUSE ON HWY|THROWING',
    'MEDICAL'        : r'MEDICAL EXAMIN|IGNITION INTERLOCK|ADDRESS CHANGE',
}

 - Precompile + categorize:

In [178]:
# ============================================================
# DESCRIPTION COLUMN - CLEANING + CATEGORIZATION
# ============================================================

# Precompile regex patterns
compiled_categories = {
    cat: re.compile(pattern, re.IGNORECASE)
    for cat, pattern in categories.items()
}

# Priority order — first match wins
priority_order = [
    'DUI', 'SPEED', 'PHONE', 'SEATBELT', 'RECKLESS',
    'HIT AND RUN', 'FRAUD', 'LICENSE', 'REGISTRATION',
    'INSURANCE', 'EQUIPMENT', 'RED LIGHT', 'STOP SIGN',
    'YIELD', 'LANE', 'FOLLOWING', 'IMPROPER TURN',
    'PARKING', 'MEDICAL',
]

# Vectorized categorization
def categorize_vectorized(series):
    result = pd.Series('OTHER', index=series.index)
    text = series.str.upper().fillna('')
    for cat in reversed(priority_order):
        pattern = compiled_categories[cat]
        mask = text.str.contains(pattern, regex=True, na=False)
        result[mask] = cat
    return result

# Clean description + create category column
traffic_df['description_clean'] = (
    traffic_df['description']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
)

traffic_df['violation_category'] = categorize_vectorized(
    traffic_df['description']
)

— Validation:

In [179]:
# ============================================================
# DESCRIPTION COLUMN - VALIDATION
# ============================================================

print(f"Total rows:    {traffic_df.shape[0]:,}")
print(f"Coverage:      {((traffic_df['violation_category'] != 'OTHER').sum()/traffic_df.shape[0]*100):.1f}%")
print(f"OTHER:         {(traffic_df['violation_category'] == 'OTHER').sum():,}")
print(f"NaN remaining: {traffic_df['description_clean'].isna().sum()}")
print(f"\nViolation category breakdown:")
print(traffic_df['violation_category'].value_counts())

Total rows:    1,048,575
Coverage:      99.5%
OTHER:         5,709
NaN remaining: 0

Violation category breakdown:
violation_category
LICENSE          244064
SPEED            210373
PHONE            110997
REGISTRATION      81708
DUI               71211
EQUIPMENT         50879
FRAUD             43247
RECKLESS          38366
LANE              37188
SEATBELT          29124
STOP SIGN         27596
YIELD             23117
INSURANCE         18551
RED LIGHT         17821
HIT AND RUN       12343
IMPROPER TURN     11767
PARKING            6835
OTHER              5709
FOLLOWING          4866
MEDICAL            2813
Name: count, dtype: int64


**Note — two new columns created:**
```
description_clean   — normalized text (strip + upper)
violation_category  — categorized violation type


#### 5.4.2 violation_type

In [180]:
print(f"NaN count: {traffic_df['violation_type'].isna().sum():,}")
print(f"Unique : {traffic_df['violation_type'].nunique():,}")
print(traffic_df['violation_type'].value_counts())


NaN count: 0
Unique : 4
violation_type
CITATION    849082
WARNING     189601
ESERO         9803
SERO            89
Name: count, dtype: int64


- Step1: Cleaning

In [181]:
# ============================================================
# VIOLATION TYPE COLUMN CLEANING
# ============================================================

traffic_df['violation_type'] = (
    traffic_df['violation_type']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
)


- Satep2: Validation

In [182]:

# Validation
print(traffic_df['violation_type'].value_counts())
print(f"\nNaN remaining: {traffic_df['violation_type'].isna().sum()}")


violation_type
CITATION    849082
WARNING     189601
ESERO         9803
SERO            89
Name: count, dtype: int64

NaN remaining: 0


#### 5.4.3 charge

In [183]:
# ============================================================
# CHARGE COLUMN CLEANING
# ============================================================

traffic_df['charge'] = (
    traffic_df['charge']
    .str.strip()
    .str.upper()
    .fillna('UNKNOWN')
)

# Validation
print(f"Total rows:     {traffic_df.shape[0]:,}")
print(f"Unique charges: {traffic_df['charge'].nunique():,}")
print(f"NaN remaining:  {traffic_df['charge'].isna().sum()}")
print(f"\nTop 10 charges:")
print(traffic_df['charge'].value_counts().head(10))

Total rows:     1,048,575
Unique charges: 1,044
NaN remaining:  0

Top 10 charges:
charge
21-801.1         171071
21-201(A1)        70435
13-409(B)         38627
16-112(C)         36025
16-303(C)         35451
16-303(H)         33866
13-401(H)         30424
16-101(A)         29617
21-1124.2(D2)     25254
13-411(F)         24584
Name: count, dtype: int64


In [184]:
# Confirm structure
print(f"Total rows:      {traffic_df.shape[0]:,}")
print(f"Unique seq_ids:  {traffic_df['seq_id'].nunique():,}")
print(f"Unique stop_ids: {traffic_df['stop_id'].nunique():,}")

# How many stops have multiple charges?
charges_per_stop = traffic_df.groupby('seq_id')['charge'].count()
print(f"\nStops with 1 charge:  {(charges_per_stop == 1).sum():,}")
print(f"Stops with 2 charges: {(charges_per_stop == 2).sum():,}")
print(f"Stops with 3 charges: {(charges_per_stop == 3).sum():,}")
print(f"Stops with 4+ charges:{(charges_per_stop >= 4).sum():,}")

# Show example of multi-charge stop
multi_charge = charges_per_stop[charges_per_stop > 1].index[0]
print(f"\nExample multi-charge stop (seq_id: {multi_charge}):")
print(traffic_df[traffic_df['seq_id'] == multi_charge][['seq_id', 'stop_id', 'charge', 'description']].to_string())

Total rows:      1,048,575
Unique seq_ids:  568,057
Unique stop_ids: 568,317

Stops with 1 charge:  388,327
Stops with 2 charges: 78,581
Stops with 3 charges: 33,523
Stops with 4+ charges:67,626

Example multi-charge stop (seq_id: 0000a11f-80e6-4dd4-8128-b199929f76a9):
                                      seq_id  stop_id      charge                                                                         description
539809  0000a11f-80e6-4dd4-8128-b199929f76a9   278049   13-401(H)                              DRIVING VEHICLE ON HIGHWAY WITH SUSPENDED REGISTRATION
552143  0000a11f-80e6-4dd4-8128-b199929f76a9   278049   13-411(F)                           DISPLAYING EXPIRED REGISTRATION PLATE ISSUED BY ANY STATE
577839  0000a11f-80e6-4dd4-8128-b199929f76a9   278049  13-401(B1)                                     OPERATING UNREGISTERED MOTOR VEHICLE ON HIGHWAY
582234  0000a11f-80e6-4dd4-8128-b199929f76a9   278049   13-411(D)  DRIVING VEHICLE ON HIGHWAY WITHOUT CURRENT REGISTRATION PLATES 

#### 5.4.4 article

In [185]:
# ============================================================
# ARTICLE COLUMN CLEANING
# ============================================================

article_map = {
    'TRANSPORTATION ARTICLE' : 'Transportation Article',
    'MARYLAND RULES'         : 'Maryland Rules',
    'BR'                     : 'Business Regulation Article',
    'TG'                     : 'Tax General Article',
    '0'                      : 'Business Regulation Article',
    '1A'                     : 'Transportation Article',
    'NAN'                    : 'Local Code',
}

traffic_df['article_clean'] = (
    traffic_df['article']
    .str.strip()
    .fillna('UNKNOWN')
    .replace(article_map)
)


In [186]:

# Validation
print(traffic_df['article_clean'].value_counts())
print(f"\nNaN remaining: {traffic_df['article_clean'].isna().sum()}")


article_clean
Transportation Article         1032892
Local Code                        9892
Maryland Rules                    5757
Business Regulation Article         28
Tax General Article                  6
Name: count, dtype: int64

NaN remaining: 0


```
1. Normalized    — strip + title case
2. Fixed NAN     — string 'NAN' → 'Local Code'
3. Decoded       — BR → Business Regulation Article
                   TG → Tax General Article
                   0  → Business Regulation Article
                   1A → Transportation Article
4. Inferred      — NAN charges are local equipment codes

#### 5.4.5 search_arrest_reason

In [187]:
print(f"NaN count: {traffic_df['search_arrest_reason'].isna().sum():,}")
print(f"Unique : {traffic_df['search_arrest_reason'].nunique():,}")
print(traffic_df['search_arrest_reason'].value_counts())

NaN count: 0
Unique : 9
search_arrest_reason
NAN          995840
STOP          36753
SEARCH         7108
OTHER          6211
WARRANT        2616
TRAFFIC          27
DUI              13
MARIHUANA         5
DWI               2
Name: count, dtype: int64


In [188]:
# Check what NAN actually means here
print(traffic_df[traffic_df['search_arrest_reason'] == 'NAN'][['search_arrest_reason', 'search_conducted', 'arrest_type']].head(5).to_string())

# Check MARIHUANA
print(traffic_df[traffic_df['search_arrest_reason'] == 'MARIHUANA'][['search_arrest_reason', 'charge', 'description']].head(3).to_string())

  search_arrest_reason search_conducted        arrest_type
0                  NAN               No  A - Marked Patrol
1                  NAN               No  A - Marked Patrol
2                  NAN               No  A - Marked Patrol
3                  NAN               No  A - Marked Patrol
4                  NAN               No  A - Marked Patrol
       search_arrest_reason     charge                                                                                        description
675092            MARIHUANA  16-303(C)  PERSON DRIVING MOTOR VEHICLE ON HIGHWAY OR PUBLIC USE PROPERTY ON SUSPENDED LICENSE AND PRIVILEGE
677751            MARIHUANA  16-101(A)   DRIVING, ATTEMPTING TO DRIVE MOTOR VEHICLE ON HIGHWAY WITHOUT REQUIRED LICENSE AND AUTHORIZATION
698423            MARIHUANA  16-112(C)          FAILURE OF INDIVIDUAL DRIVING ON HIGHWAY TO DISPLAY LICENSE TO UNIFORMED POLICE ON DEMAND


In [189]:
# ============================================================
# SEARCH ARREST REASON COLUMN CLEANING
# ============================================================

search_arrest_map = {
    'NAN'       : 'NOT APPLICABLE',
    'MARIHUANA' : 'MARIJUANA',
}

traffic_df['search_arrest_reason_clean'] = (
    traffic_df['search_arrest_reason']
    .str.strip()
    .str.upper()
    .fillna('NOT APPLICABLE')
    .replace(search_arrest_map)
)


In [190]:

# Validation
print(traffic_df['search_arrest_reason_clean'].value_counts())
print(f"\nNaN remaining: {traffic_df['search_arrest_reason_clean'].isna().sum()}")

search_arrest_reason_clean
NOT APPLICABLE    995840
STOP               36753
SEARCH              7108
OTHER               6211
WARRANT             2616
TRAFFIC               27
DUI                   13
MARIJUANA              5
DWI                    2
Name: count, dtype: int64

NaN remaining: 0


In [191]:
traffic_df.head(2)

,seq_id,date_of_stop,time_of_stop,agency,sub_agency,description,location,latitude,longitude,accident,belts,personal_injury,property_damage,fatal,commercial_license,hazmat,commercial_vehicle,alcohol,work_zone,search_conducted,search_disposition,search_outcome,search_reason,search_reason_for_stop,search_type,search_arrest_reason,state,vehicle_type,year,make,model,color,violation_type,charge,article,contributed_to_accident,race,gender,driver_city,driver_state,dl_state,arrest_type,stop_timestamp,district_number,direction,exit_ref,road_1,road_2,loc_type,is_landmark,needs_review,location_clean,is_geo_outlier,stop_id,vehicle_code,vehicle_category,make_clean,model_clean,color_clean,driver_city_clean,description_clean,violation_category,article_clean,search_arrest_reason_clean
0,52282e8c-f2e1-4bb5-8509-2d5e4f8da8ca,2023-01-05,23:11:00,MCP,"3rd District, Silver Spring",OPERATING UNREGISTERED MOTOR VEHICLE ON HIGHWAY,BRIGGS CHANEY RD @ COLUMIBA PIKE,NaN,NaN,0,0,0,0,0,0,0,0,0,0,No,NaN,Citation,NaN,17-107(a1),NaN,NAN,MD,02 - AUTOMOBILE,2007,CHEV,CRUZ,BLACK,CITATION,13-401(B1),TRANSPORTATION ARTICLE,0,WHITE,M,GAITHERSBURG,MD,MD,A - Marked Patrol,2023-01-05 23:11:00,3,NaN,NaN,BRIGGS CHANEY RD,COLUMIBA PIKE,intersection,False,False,BRIGGS CHANEY RD/COLUMIBA PIKE,False,1,02,AUTOMOBILE,CHEVROLET,CRUZ,BLACK,GAITHERSBURG,OPERATING UNREGISTERED MOTOR VEHICLE ON HIGHWAY,REGISTRATION,Transportation Article,NOT APPLICABLE
1,b66f253b-af29-4bc4-bb73-93755ca2a779,2023-08-31,16:41:00,MCP,"6th District, Gaithersburg / Montgomery Village",DRIVING TO DRIVE MOTOR VEHICLE ON HIGHWAY WITH...,OAKMONT AVE @ GROVEMONT CIR,39.097965,-77.15301,0,0,0,0,0,0,0,0,0,0,No,NaN,Citation,NaN,20-102(a1),NaN,NAN,MD,02 - AUTOMOBILE,2005,FORD,EXPLORER,BLACK,CITATION,16-101(A1),TRANSPORTATION ARTICLE,0,HISPANIC,M,GAITHERSBURG,MD,MD,A - Marked Patrol,2023-08-31 16:41:00,6,NaN,NaN,OAKMONT AVE,GROVEMONT CIR,intersection,False,False,OAKMONT AVE/GROVEMONT CIR,False,2,02,AUTOMOBILE,FORD,EXPLORER,BLACK,GAITHERSBURG,DRIVING TO DRIVE MOTOR VEHICLE ON HIGHWAY WITH...,LICENSE,Transportation Article,NOT APPLICABLE


#### 5.4.6 violation_charge_df Table Creation

- Create a table with:

| Column                     | Reason          |
| -------------------------- | --------------- |
| stop_id                    | FK              |
| description_clean          | human readable  |
| violation_type             | category        |
| violation_category         | good enrichment |
| charge                     | legal code      |
| article_clean              | standardized    |
| search_arrest_reason_clean | context         |



In [192]:
violation_charge_cols = [
    'stop_id',
    'description_clean',
    'violation_type',
    'violation_category',
    'charge',
    'article_clean',
    'search_arrest_reason_clean'
]

- Step1: Create dataframe

In [193]:
violation_charge_df = traffic_df[violation_charge_cols].copy()

- Step2: Rename

In [194]:
violation_charge_df.rename(columns={
    'description_clean': 'description',
    'article_clean': 'article',
    'search_arrest_reason_clean': 'search_arrest_reason'
}, inplace=True)

- Step 3: Drop duplicates

In [195]:
violation_charge_df = violation_charge_df.drop_duplicates().reset_index(drop=True)

- Step 4: Add primary key

In [196]:
violation_charge_df.insert(0, 'charge_id', range(1, len(violation_charge_df)+1))

In [197]:
violation_charge_df.head()

,charge_id,stop_id,description,violation_type,violation_category,charge,article,search_arrest_reason
0,1,1,OPERATING UNREGISTERED MOTOR VEHICLE ON HIGHWAY,CITATION,REGISTRATION,13-401(B1),Transportation Article,NOT APPLICABLE
1,2,2,DRIVING TO DRIVE MOTOR VEHICLE ON HIGHWAY WITH...,CITATION,LICENSE,16-101(A1),Transportation Article,NOT APPLICABLE
2,3,2,FAILURE TO DISPLAY REGISTRATION CARD UPON DEMA...,CITATION,FRAUD,13-409(B),Transportation Article,NOT APPLICABLE
3,4,2,DRIVER OF MOTOR VEHICLE FOLLOWING VEHICLE CLOS...,CITATION,FOLLOWING,21-310(A),Transportation Article,NOT APPLICABLE
4,5,2,FAILURE TO CONTROL VEH. SPEED ON HWY. TO AVOID...,CITATION,SPEED,21-801(B),Transportation Article,NOT APPLICABLE


In [198]:
# ============================================================
# VIOLATION CHARGE — DDL PREPARATION
# ============================================================

print(f"Total rows:        {violation_charge_df.shape[0]:,}")
print(f"Unique charge_ids: {violation_charge_df['charge_id'].nunique():,}")
print(f"NaN charge_ids:    {violation_charge_df['charge_id'].isna().sum():,}")
print(f"Is charge_id unique: {violation_charge_df['charge_id'].nunique() == violation_charge_df.shape[0]}")

# Check varchar lengths
str_cols = ['description', 'violation_type', 'violation_category',
            'charge', 'article', 'search_arrest_reason']
print(f"\nMax lengths:")
for col in str_cols:
    max_len = violation_charge_df[col].str.len().max()
    print(f"  {col:25} max length: {max_len}")

# Check dtypes
print(f"\nDtypes:")
print(violation_charge_df.dtypes)

# Check NaN counts
print(f"\nNaN counts:")
print(violation_charge_df.isna().sum())

Total rows:        1,046,801
Unique charge_ids: 1,046,801
NaN charge_ids:    0
Is charge_id unique: True

Max lengths:
  description               max length: 108
  violation_type            max length: 8
  violation_category        max length: 13
  charge                    max length: 14
  article                   max length: 27
  search_arrest_reason      max length: 14

Dtypes:
charge_id                int64
stop_id                  int64
description             object
violation_type          object
violation_category      object
charge                  object
article                 object
search_arrest_reason    object
dtype: object

NaN counts:
charge_id               0
stop_id                 0
description             0
violation_type          0
violation_category      0
charge                  0
article                 0
search_arrest_reason    0
dtype: int64


## 5.5 Search_Enforcement 

#### 5.5.1 search_conducted

In [199]:
print(f"NaN count: {traffic_df['search_conducted'].isna().sum():,}")
print(f"Unique : {traffic_df['search_conducted'].nunique():,}")
print(traffic_df['search_conducted'].value_counts())


NaN count: 453,740
Unique : 2
search_conducted
No     524852
Yes     69983
Name: count, dtype: int64


In [200]:
# #Check what NaN rows look like
# print(traffic_df[traffic_df['search_conducted'].isna()][
#     ['search_conducted', 'search_arrest_reason', 'search_type', 'search_outcome']
# ].head(10).to_string())

```
search_conducted search_arrest_reason search_type search_outcome
12              NaN                  NAN         NaN            NaN
14              NaN                  NAN         NaN            NaN
22              NaN                  NAN         NaN            NaN
23              NaN                  NAN         NaN            NaN
24              NaN                  NAN         NaN            NaN
25              NaN                  NAN         NaN            NaN
30              NaN                  NAN         NaN            NaN
31              NaN                  NAN         NaN            NaN
32              NaN                  NAN         NaN            NaN
36              NaN                  NAN         NaN            NaN

search_arrest_reason = NAN  (not applicable)
search_type          = NaN  (no search type)
search_outcome       = NaN  (no outcome)

- This conclusively confirms NaN = No search conducted. Safe to fill with 'No':

In [201]:
traffic_df['search_conducted_clean'] = (
    traffic_df['search_conducted']
    .str.strip()
    .fillna('No')
    .map({'Yes': 1, 'No': 0})
)

# Validation
print(traffic_df['search_conducted_clean'].value_counts())
print(f"\nNaN remaining: {traffic_df['search_conducted_clean'].isna().sum()}")
print(f"Dtype: {traffic_df['search_conducted_clean'].dtype}")


search_conducted_clean
0    978592
1     69983
Name: count, dtype: int64

NaN remaining: 0
Dtype: int64


- **Cleaning Performed:**
  - Standardized Yes/No values
  - Treated missing values as "No" (no search recorded)
  - Converted to binary format (1 = Yes, 0 = No)

- **Notes:**
  - High number of missing values interpreted as no search conducted

#### 5.5.2 search_disposition

In [202]:
print(f"NaN count: {traffic_df['search_disposition'].isna().sum():,}")
print(f"Unique : {traffic_df['search_disposition'].nunique():,}")
print(traffic_df['search_disposition'].value_counts())


NaN count: 978,592
Unique : 7
search_disposition
Nothing                    31260
Contraband Only            15452
Property Only              13060
Contraband and Property    10195
DUI                           12
marijuana                      3
nothing                        1
Name: count, dtype: int64


In [203]:
# #Verify NaN matches 'No' search_conducted
# no_search = (traffic_df['search_conducted_clean'] == 0).sum()
# nan_disposition = traffic_df['search_disposition'].isna().sum()
# print(f"No search conducted: {no_search:,}")
# print(f"NaN disposition:     {nan_disposition:,}")
# print(f"Match: {no_search == nan_disposition}")

```
No search conducted: 978,592
NaN disposition:     978,592
Match: True
- match confirmed! Now apply the cleaning:

In [204]:
print(traffic_df[traffic_df['search_disposition'] == 'marijuana'][
    ['date_of_stop', 'search_disposition', 'charge', 'description']
].to_string())

       date_of_stop search_disposition     charge                                                                                        description
541731   2021-10-25          marijuana  16-303(C)  PERSON DRIVING MOTOR VEHICLE ON HIGHWAY OR PUBLIC USE PROPERTY ON SUSPENDED LICENSE AND PRIVILEGE
575643   2021-10-25          marijuana  22-609(B)                                       MOTOR VEH. EQUIPPED WITH UNLAWFULLY MODIFIED EXHAUST SYSTEM.
610782   2021-10-25          marijuana  16-301(J)                                                                          POSSESSING SUSPENDED LIC.


- All 3 stops are from October 25, 2021 — well before Maryland legalization in July 2023. 
- So marijuana was illegal at that time → should be 'Contraband Only':

In [205]:
disposition_map = {
    'Nothing'   : 'Nothing Found',
    'nothing'   : 'Nothing Found',
    'marijuana' : 'Contraband Only',  # pre-legalization, illegal
    'DUI'       : 'DUI',
}

traffic_df['search_disposition_clean'] = (
    traffic_df['search_disposition']
    .str.strip()
    .fillna('Not Applicable')
    .replace(disposition_map)
)

# Validation
print(traffic_df['search_disposition_clean'].value_counts())
print(f"\nNaN remaining: {traffic_df['search_disposition_clean'].isna().sum()}")

search_disposition_clean
Not Applicable             978592
Nothing Found               31261
Contraband Only             15455
Property Only               13060
Contraband and Property     10195
DUI                            12
Name: count, dtype: int64

NaN remaining: 0


```
1. Normalized    — strip + fillna
2. Fixed case    — nothing → Nothing Found
                   marijuana → Contraband Only
3. Filled NaN    — confirmed matches no search rows
4. Renamed       — Nothing → Nothing Found (clearer)
5. Merged        — marijuana → Contraband Only
                   (pre-legalization, was illegal)

#### 5.5.3 search_outcome

In [206]:
print(f"NaN count: {traffic_df['search_outcome'].isna().sum():,}")
print(f"Unique : {traffic_df['search_outcome'].nunique():,}")
print(traffic_df['search_outcome'].value_counts())


NaN count: 472,821
Unique : 5
search_outcome
Citation              428401
Warning                88931
Arrest                 53537
SERO                    4882
Recovered Evidence         3
Name: count, dtype: int64


In [207]:
# Check searches with no outcome
print(traffic_df[
    (traffic_df['search_conducted_clean'] == 1) & 
    (traffic_df['search_outcome'].isna())
][['search_conducted_clean', 'search_outcome', 'search_disposition_clean']].value_counts())

Series([], Name: count, dtype: int64)


In [208]:
# Check what NaN outcome rows look like
print(traffic_df[traffic_df['search_outcome'].isna()][
    ['search_conducted_clean', 'search_disposition_clean', 'search_outcome']
].value_counts().head(10))

Series([], Name: count, dtype: int64)


In [209]:
# Check the actual distribution of search_conducted for NaN outcome rows
print(traffic_df[traffic_df['search_outcome'].isna()][
    'search_conducted_clean'
].value_counts())

# Check date range of NaN outcome rows
print(f"\nDate range of NaN outcome rows:")
print(traffic_df[traffic_df['search_outcome'].isna()]['date_of_stop'].describe())

search_conducted_clean
0    472821
Name: count, dtype: int64

Date range of NaN outcome rows:
count                           472821
mean     2016-05-19 08:30:59.856478464
min                2012-01-01 00:00:00
25%                2013-05-17 00:00:00
50%                2015-05-22 00:00:00
75%                2018-09-15 00:00:00
max                2025-12-08 00:00:00
Name: date_of_stop, dtype: object


- No search conducted → no search outcome recorded
- NaN = Not Applicable ✅

In [210]:
# ============================================================
# SEARCH OUTCOME COLUMN CLEANING
# ============================================================

traffic_df['search_outcome_clean'] = (
    traffic_df['search_outcome']
    .str.strip()
    .fillna('Not Applicable')
)

# Validation
print(traffic_df['search_outcome_clean'].value_counts())
print(f"\nNaN remaining: {traffic_df['search_outcome_clean'].isna().sum()}")

search_outcome_clean
Not Applicable        472821
Citation              428401
Warning                88931
Arrest                 53537
SERO                    4882
Recovered Evidence         3
Name: count, dtype: int64

NaN remaining: 0


```

1. Strip whitespace
2. Fill NaN → 'Not Applicable'
   (confirmed: all NaN rows had no search conducted)

search_conducted    → 69,983  searches (6.7%)
search_disposition  → 69,983  outcomes
search_outcome      → 69,983  recorded

#### 5.5.4 search_reason

In [211]:
print(f"NaN count: {traffic_df['search_reason'].isna().sum():,}")
print(f"Unique : {traffic_df['search_reason'].nunique():,}")
print(traffic_df['search_reason'].value_counts())


NaN count: 978,592
Unique : 9
search_reason
Incident to Arrest        46934
Probable Cause            12849
Consensual                 7794
K-9                        1013
Other                       933
Exigent Circumstances       452
Probable Cause for CDS        4
Arrest/Tow                    3
plain view marijuana          1
Name: count, dtype: int64


In [212]:
print(f"No search conducted: {(traffic_df['search_conducted_clean'] == 0).sum():,}")
print(f"NaN search_reason:   {traffic_df['search_reason'].isna().sum():,}")
print(f"Match: {(traffic_df['search_conducted_clean'] == 0).sum() == traffic_df['search_reason'].isna().sum()}")

No search conducted: 978,592
NaN search_reason:   978,592
Match: True


In [213]:
# ============================================================
# SEARCH REASON COLUMN CLEANING
# ============================================================

search_reason_map = {
    'plain view marijuana'   : 'Plain View',
    'Probable Cause for CDS' : 'Probable Cause',
    'Arrest/Tow'             : 'Incident to Arrest',
}

traffic_df['search_reason_clean'] = (
    traffic_df['search_reason']
    .str.strip()
    .fillna('Not Applicable')
    .replace(search_reason_map)
)

# Validation
print(traffic_df['search_reason_clean'].value_counts())
print(f"\nNaN remaining: {traffic_df['search_reason_clean'].isna().sum()}")


search_reason_clean
Not Applicable           978592
Incident to Arrest        46937
Probable Cause            12853
Consensual                 7794
K-9                        1013
Other                       933
Exigent Circumstances       452
Plain View                    1
Name: count, dtype: int64

NaN remaining: 0


```

1. Strip + fillna    — NaN → Not Applicable
2. Merged            — Probable Cause for CDS → Probable Cause
                       Arrest/Tow → Incident to Arrest
3. Standardized      — plain view marijuana → Plain View

search_conducted   → 69,983 searches
search_reason      → 69,983 reasons recorded
search_disposition → 69,983 dispositions
search_outcome     → 69,983 outcomes

#### 5.5.5 search_reason_for_stop

In [214]:
print(f"NaN count: {traffic_df['search_reason_for_stop'].isna().sum():,}")
print(f"Unique : {traffic_df['search_reason_for_stop'].nunique():,}")
print(traffic_df['search_reason_for_stop'].value_counts())


NaN count: 453,977
Unique : 707
search_reason_for_stop
21-201(a1)       70497
21-801.1         43863
13-411(f)        27523
13-401(h)        25399
21-1124.2(d2)    24026
                 ...  
16-810(d)            1
13-921(d2)           1
8-409(e)             1
22-228(b)            1
23-302               1
Name: count, Length: 707, dtype: int64


In [215]:
# Check if search_reason_for_stop matches charge column
print("Same as charge column?")
print(f"Unique charge codes:                {traffic_df['charge'].nunique():,}")
print(f"Unique search_reason_for_stop:      {traffic_df['search_reason_for_stop'].nunique():,}")

# Check overlap
reason_codes = set(traffic_df['search_reason_for_stop'].dropna().str.upper().str.strip())
charge_codes = set(traffic_df['charge'].dropna())
print(f"\nCodes in both:     {len(reason_codes & charge_codes):,}")
print(f"Only in reason:    {len(reason_codes - charge_codes):,}")
print(f"Only in charge:    {len(charge_codes - reason_codes):,}")

# Check NaN pattern
print(f"\nNaN search_reason_for_stop: {traffic_df['search_reason_for_stop'].isna().sum():,}")
print(f"No search conducted:        {(traffic_df['search_conducted_clean'] == 0).sum():,}")

Same as charge column?
Unique charge codes:                1,044
Unique search_reason_for_stop:      707

Codes in both:     699
Only in reason:    4
Only in charge:    345

NaN search_reason_for_stop: 453,977
No search conducted:        978,592


In [216]:
# Why don't NaNs match search_conducted?
print(traffic_df[
    (traffic_df['search_conducted_clean'] == 1) & 
    (traffic_df['search_reason_for_stop'].isna())
]['search_reason_for_stop'].count())

# Check what the 4 unique codes are
unique_codes = reason_codes - charge_codes
print(f"\n4 codes only in search_reason_for_stop:")
for code in unique_codes:
    print(f"  {code}")
    print(traffic_df[traffic_df['search_reason_for_stop'].str.upper().str.strip() == code][
        ['search_reason_for_stop', 'charge', 'description']].head(2).to_string())

0

4 codes only in search_reason_for_stop:
  13-411
       search_reason_for_stop     charge                                                                                description
95331                  13-411  14-107(K)                 ATTACHING UNAUTHORIZED VEH. REG. PLATE WITH INTENTTO CONCEAL VEH. OWNER ID
111391                 13-411  16-303(H)  PERSON DRIVING MOTOR VEHICLE WHILE LICENSE SUSPENDED UNDER 17-106, 26-204, 26-206, 27-103
  21-1412(D)
       search_reason_for_stop     charge                                                                                        description
116611             21-1412(d)  16-303(C)  PERSON DRIVING MOTOR VEHICLE ON HIGHWAY OR PUBLIC USE PROPERTY ON SUSPENDED LICENSE AND PRIVILEGE
122911             21-1412(d)  16-101(A)   DRIVING, ATTEMPTING TO DRIVE MOTOR VEHICLE ON HIGHWAY WITHOUT REQUIRED LICENSE AND AUTHORIZATION
  13-420(B)
        search_reason_for_stop     charge                                                                  

```a
13-411    → 13-411(F) or 13-411(D)  (missing subsection)
13-420(B) → 13-420(b)               (case difference)
21-1412(D)→ 21-1412(d)              (case difference)
55*-      → 55*                     (extra dash)

In [217]:
# ============================================================
# SEARCH REASON FOR STOP COLUMN CLEANING
# ============================================================

traffic_df['search_reason_for_stop_clean'] = (
    traffic_df['search_reason_for_stop']
    .str.strip()
    .str.upper()
    .str.rstrip('-')        # remove trailing dash e.g. 55*-
    .fillna('Not Applicable')
)

# Validation
print(f"Total rows:      {traffic_df.shape[0]:,}")
print(f"Unique codes:    {traffic_df['search_reason_for_stop_clean'].nunique():,}")
print(f"NaN remaining:   {traffic_df['search_reason_for_stop_clean'].isna().sum()}")
print(f"\nTop 10:")
print(traffic_df['search_reason_for_stop_clean'].value_counts().head(10))

Total rows:      1,048,575
Unique codes:    703
NaN remaining:   0

Top 10:
search_reason_for_stop_clean
Not Applicable    453977
21-201(A1)         70497
21-801.1           43863
13-411(F)          27523
13-401(H)          25399
21-1124.2(D2)      24026
21-707(A)          23117
16-303(C)          20098
21-801(B)          19889
21-202(H1)         13498
Name: count, dtype: int64


#### 5.5.6 search_type

In [218]:
print(f"NaN count: {traffic_df['search_type'].isna().sum():,}")
print(f"Unique : {traffic_df['search_type'].nunique():,}")
print(traffic_df['search_type'].value_counts())


NaN count: 978,599
Unique : 6
search_type
Both                 52328
Person                9594
Property              8046
car                      4
Search Incidental        3
PC                       1
Name: count, dtype: int64


In [219]:
print(f"No search conducted: {(traffic_df['search_conducted_clean'] == 0).sum():,}")
print(f"NaN search_type:     {traffic_df['search_type'].isna().sum():,}")
print(f"Match: {(traffic_df['search_conducted_clean'] == 0).sum() == traffic_df['search_type'].isna().sum()}")

No search conducted: 978,592
NaN search_type:     978,599
Match: False


In [220]:
# Find the 7 rows where search was conducted but search_type is NaN
print(traffic_df[
    (traffic_df['search_conducted_clean'] == 1) & 
    (traffic_df['search_type'].isna())
][['search_conducted_clean', 'search_type', 'search_reason_clean', 'search_disposition_clean']].to_string())

        search_conducted_clean search_type search_reason_clean search_disposition_clean
265548                       1         NaN      Probable Cause          Contraband Only
276590                       1         NaN      Probable Cause          Contraband Only
284574                       1         NaN      Probable Cause          Contraband Only
299784                       1         NaN      Probable Cause          Contraband Only
311108                       1         NaN      Probable Cause          Contraband Only
316458                       1         NaN      Probable Cause          Contraband Only
337841                       1         NaN      Probable Cause          Contraband Only


In [221]:
# Check what search_type is used with Probable Cause + Contraband Only
print(traffic_df[
    (traffic_df['search_reason_clean'] == 'Probable Cause') &
    (traffic_df['search_disposition_clean'] == 'Contraband Only') &
    (traffic_df['search_type'].notna())
]['search_type'].value_counts())

search_type
Both        5546
Property    1018
Person        35
Name: count, dtype: int64


In [222]:
# ============================================================
# SEARCH TYPE COLUMN CLEANING
# ============================================================

search_type_map = {
    'CAR'               : 'PROPERTY',
    'SEARCH INCIDENTAL' : 'INCIDENT TO ARREST',
    'PC'                : 'PROBABLE CAUSE',
    'BOTH'              : 'BOTH',
    'PERSON'            : 'PERSON',
    'PROPERTY'          : 'PROPERTY',
}

traffic_df['search_type_clean'] = (
    traffic_df['search_type']
    .str.strip()
    .str.upper()
    .fillna('NOT APPLICABLE')
    .replace(search_type_map)
)

# Validation
print(traffic_df['search_type_clean'].value_counts())
print(f"\nNaN remaining: {traffic_df['search_type_clean'].isna().sum()}")


search_type_clean
NOT APPLICABLE        978599
BOTH                   52328
PERSON                  9594
PROPERTY                8050
INCIDENT TO ARREST         3
PROBABLE CAUSE             1
Name: count, dtype: int64

NaN remaining: 0


1. Uppercase         — all values normalized
2. Fill NaN          → NOT APPLICABLE
3. Fixed 7 rows      → inferred 'BOTH' from similar rows
4. Standardized      — CAR → PROPERTY
                       SEARCH INCIDENTAL → INCIDENT TO ARREST
                       PC → PROBABLE CAUSE

#### 5.5.7 arrest_type

In [223]:
print(f"NaN count: {traffic_df['arrest_type'].isna().sum():,}")
print(f"Unique : {traffic_df['arrest_type'].nunique():,}")
print(traffic_df['arrest_type'].value_counts())


NaN count: 0
Unique : 19
arrest_type
A - Marked Patrol                         785542
Q - Marked Laser                          127824
B - Unmarked Patrol                        55943
S - License Plate Recognition              15011
G - Marked Moving Radar (Stationary)       14694
L - Motorcycle                             13608
O - Foot Patrol                            11790
E - Marked Stationary Radar                 8794
R - Unmarked Laser                          7032
I - Marked Moving Radar (Moving)            3063
M - Marked (Off-Duty)                       2086
H - Unmarked Moving Radar (Stationary)       921
F - Unmarked Stationary Radar                846
J - Unmarked Moving Radar (Moving)           465
C - Marked VASCAR                            319
N - Unmarked (Off-Duty)                      230
D - Unmarked VASCAR                          229
P - Mounted Patrol                           144
K - Aircraft Assist                           34
Name: count, dtype: int64


In [224]:
# ============================================================
# ARREST TYPE COLUMN CLEANING
# ============================================================

# Normalize first
traffic_df['arrest_type_clean'] = (
    traffic_df['arrest_type']
    .str.strip()
    .str.upper()
)

# Split into code and description
traffic_df['arrest_type_code'] = (
    traffic_df['arrest_type_clean']
    .str.extract(r'^([A-Z])\s*-')
)

traffic_df['arrest_type_desc'] = (
    traffic_df['arrest_type_clean']
    .str.extract(r'^[A-Z]\s*-\s*(.+)$')
)

# Validation
print("Code distribution:")
print(traffic_df['arrest_type_code'].value_counts())
print(f"\nDescription distribution:")
print(traffic_df['arrest_type_desc'].value_counts())
print(f"\nNaN in code: {traffic_df['arrest_type_code'].isna().sum()}")
print(f"NaN in desc: {traffic_df['arrest_type_desc'].isna().sum()}")


Code distribution:
arrest_type_code
A    785542
Q    127824
B     55943
S     15011
G     14694
L     13608
O     11790
E      8794
R      7032
I      3063
M      2086
H       921
F       846
J       465
C       319
N       230
D       229
P       144
K        34
Name: count, dtype: int64

Description distribution:
arrest_type_desc
MARKED PATROL                         785542
MARKED LASER                          127824
UNMARKED PATROL                        55943
LICENSE PLATE RECOGNITION              15011
MARKED MOVING RADAR (STATIONARY)       14694
MOTORCYCLE                             13608
FOOT PATROL                            11790
MARKED STATIONARY RADAR                 8794
UNMARKED LASER                          7032
MARKED MOVING RADAR (MOVING)            3063
MARKED (OFF-DUTY)                       2086
UNMARKED MOVING RADAR (STATIONARY)       921
UNMARKED STATIONARY RADAR                846
UNMARKED MOVING RADAR (MOVING)           465
MARKED VASCAR                       

```
arrest_type_clean  → 'A - MARKED PATROL'  (full normalized)
arrest_type_code   → 'A'                  (for filtering/grouping)
arrest_type_desc   → 'MARKED PATROL'      (for display/analysis)

```
MARKED PATROL        785,542  (74.9%)  → regular marked cars
MARKED LASER         127,824  (12.2%)  → speed guns
UNMARKED PATROL       55,943   (5.3%)  → unmarked cars
LICENSE PLATE RECOGNITION 15,011       → LPR technology
RADAR combined        28,469           → various radar types
MOTORCYCLE            13,608
FOOT PATROL           11,790

#### 5.5.8 search_enforcement_df Table Creation

In [225]:
search_enforcement_cols = [
    'stop_id', 'driver_id', 'search_conducted_clean', 'search_disposition_clean',
    'search_outcome_clean', 'search_reason_clean', 'search_reason_for_stop_clean',
    'search_type_clean', 'arrest_type_code', 'arrest_type_desc'
]

search_enforcement_df = (
    traffic_df
    .merge(
        driver_vehicle_df[['stop_id', 'driver_id']],  # only bring driver_id
        on='stop_id',
        how='left'
    )
    [search_enforcement_cols]
)

In [226]:
search_enforcement_df.columns = search_enforcement_df.columns.str.replace('_clean', '', regex=False)

In [227]:
print(traffic_df.shape[0] == search_enforcement_df.shape[0])  # should be True
print(search_enforcement_df['driver_id'].isna().sum())         # should be 0

True
0


In [228]:
# Deduplicate — keep only one row per stop
search_enforcement_clean = search_enforcement_df.drop_duplicates(
    subset=['stop_id', 'driver_id']
).reset_index(drop=True)

# Verify
print(f"Before: {search_enforcement_df.shape[0]:,}")
print(f"After:  {search_enforcement_clean.shape[0]:,}")
print(f"Unique stop_ids: {search_enforcement_clean['stop_id'].nunique():,}")
print(f"Is stop_id unique: {search_enforcement_clean['stop_id'].nunique() == search_enforcement_clean.shape[0]}")


Before: 1,048,575
After:  568,317
Unique stop_ids: 568,317
Is stop_id unique: True


In [229]:
traffic_stop_df.to_csv("../data/traffic_stop_v1.csv", index=False)
driver_vehicle_df.to_csv("../data/driver_vehicle_v1.csv", index=False)
violation_charge_df.to_csv("../data/violation_charge_v1.csv", index=False)
incident_safety_df.to_csv("../data/incident_safety_v1.csv", index=False)
search_enforcement_df.to_csv("../data/search_enforcement_v1.csv", index=False)